# Rice → Pearl Millet Cross-Crop Transfer Learning — End-to-End Training Architecture

## Research question
Does representation learning on a large, labelled **rice** disease corpus (10,407 images, 10 classes) improve disease classification on a **small pearl-millet** corpus (228 images, 5 classes) compared with (a) training from scratch and (b) ImageNet initialisation alone?

## Pipeline

```
                      ┌────────────────────── DATA LAYER ───────────────────────┐
 Rice train.csv ─┐    │ audit-driven exclusions → leakage-safe grouped split    │
 Rice audit CSVs ┼──► │ (train/val/test, near-dup clusters never straddle)      │
                 │    └──────────────┬──────────────────────────────────────────┘
 Pearl images ───┘    Pearl: 512px cache → pHash groups → StratifiedGroupKFold(5)
                                      │
   STAGE 0  init ∈ {random, ImageNet}                         ┌───────────────────────────┐
        │                                                     │ MODEL: CropDiseaseNet     │
        ▼                                                     │  ConvNeXt-Tiny backbone   │
   STAGE 1  supervised pre-training on RICE  ──checkpoint──►  │  (8 blocks, 768-d GAP)    │
        (head_rice, 10 classes, val macro-F1 early stop)      │  + LayerNorm + Dropout    │
        │                                                     │  + swappable linear heads │
        ▼                                                     └───────────────────────────┘
   STAGE 2  transfer to PEARL MILLET (new head_pearl, 5 classes)
        strategies: linear_probe │ partial_ft │ full_ft (layer-wise LR decay)
        baselines : scratch │ imagenet-only ;  data-fraction curves 10/25/50/100 %
        protocol  : 5-fold grouped CV × seeds, fixed epochs (no test-fold selection)
        │
        ▼
   STAGE 3  evaluation & analysis
        macro-F1 / balanced-acc / per-class F1 / confusion / ECE, paired comparison,
        embedding (domain-gap) plot, Grad-CAM
```

## Design decisions (and why)
| Decision | Rationale |
|---|---|
| ConvNeXt-Tiny (torchvision) | Strong ImageNet transfer, 28 M params, fits RTX 5090 at 320 px with bf16. `timm` is not installed, so only torchvision is used. |
| Separate heads per domain | Rice (10) and pearl millet (5) label spaces differ; only the backbone is transferred. |
| Group-aware splits | The audit found hundreds of pHash near-duplicates (distance ≤ 2) in rice; random splitting would leak them across train/test and inflate scores. |
| Exclusions from the audit | Exact duplicates, user-confirmed cross-class conflicts (both images removed), user-confirmed same-class duplicates (one copy removed). Unreviewed same-class pairs are not deleted, but are **grouped** so they stay on one side of a split. |
| Fixed-epoch protocol for pearl millet | 228 images are too few to carve out a validation set per fold without starving classes (smallest class = 9). Selecting epochs on the test fold would leak. |
| Class-balanced loss (effective number) + label smoothing | Rice ranges 337–1,764 per class; pearl millet 9–98. |
| Macro-F1 as primary metric | Accuracy is dominated by the majority class (Pyricularia ≈ 43 % of pearl millet). |
| Resumable results log | Experiment grid is long; each finished run is appended to `results.csv` and skipped on re-run. |

## Scientific caveats built into the design
* Pearl millet is imaged in different conditions (mostly 4000×3000 field photos vs rice 480×640), so gains may reflect domain/background effects rather than lesion knowledge. The embedding and Grad-CAM analyses are there to check this.
* Rice `blast` / `downy_mildew` may be pathologically analogous to pearl-millet *Pyricularia* / *Sclerospora*; treat that as a hypothesis to test per class, not an assumption.
* With 5-fold CV on 228 images, confidence intervals are wide: always report mean ± std over folds × seeds.

Long jobs are guarded by `RUN_*` flags in the orchestration cell (all `False` by default).

In [37]:
# ============================================================
# 0. Config, paths, reproducibility
# ============================================================
import os, json, math, time, random, copy, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T, models
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, balanced_accuracy_score, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support)
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

SMOKE = os.environ.get("SMOKE", "0") == "1"       # tiny end-to-end check

# Project paths
PROJECT_DIR   = Path(os.environ.get("PEARL_PROJECT_DIR", Path.home() / "cate_model" / "Pearl_millet_Prediction"))
RICE_DIR      = PROJECT_DIR / "Rice_Dataset"
EXTRACTED_DIR = RICE_DIR / "Extracted"
IMAGE_DIR     = EXTRACTED_DIR / "train_images"
CSV_PATH      = EXTRACTED_DIR / "train.csv"
AUDIT_DIR     = RICE_DIR / "Audit"
PEARL_DIR     = PROJECT_DIR / "Pearl_Millet_Dataset" / "Extracted"
PEARL_CACHE   = PROJECT_DIR / "Pearl_Millet_Dataset" / "Cache_512"
EXP_DIR       = PROJECT_DIR / ("Experiments_smoke" if SMOKE else "Experiments")
for d in (EXP_DIR, EXP_DIR / "manifests", EXP_DIR / "checkpoints", EXP_DIR / "figures"):
    d.mkdir(parents=True, exist_ok=True)

@dataclass
class Cfg:
    backbone: str = "convnext_tiny"
    img_size: int = 320
    batch_size: int = 64
    num_workers: int = 12
    drop: float = 0.3
    label_smoothing: float = 0.1
    balance: str = "loss"            # none | loss | sampler
    mix_alpha: float = 0.0           # >0 enables mixup
    amp: bool = True                 # bf16 autocast
    seed: int = 42
    # rice pre-training
    rice_epochs: int = 15
    rice_lr: float = 4e-4
    rice_wd: float = 0.05
    rice_patience: int = 4
    group_hamming: int = 2           # pHash distance that defines a leakage group
    # pearl transfer
    pearl_epochs: int = 40
    pearl_lr: float = 2e-4
    pearl_wd: float = 0.05
    head_lr_mult: float = 10.0
    lld: float = 0.8                 # layer-wise LR decay
    n_folds: int = 5
    warmup_frac: float = 0.1

cfg = Cfg()
if SMOKE:
    cfg.img_size, cfg.batch_size, cfg.num_workers = 128, 16, 4
    cfg.rice_epochs, cfg.pearl_epochs, cfg.rice_patience = 1, 2, 1

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
seed_everything(cfg.seed)

print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("torch", torch.__version__, "| SMOKE:", SMOKE)
print("Experiments →", EXP_DIR)

Device: cuda NVIDIA GeForce RTX 5090
torch 2.14.0+cu130 | SMOKE: False
Experiments → /home/zahidrashid/cate_model/Pearl_millet_Prediction/Experiments


## 1. Data layer — Rice manifest (audit-driven, leakage-safe)

Inputs are the audit artefacts already produced by `rice_dataset_audit.ipynb`.

1. Start from `train.csv` (10,407 records).
2. **Exclude** byte-identical copies (`exclude_exact_duplicate`, 74), both images of each user-confirmed cross-class near-duplicate (label conflict), and `path_b` of each user-confirmed same-class pair (batch 01).
3. **Group** the remaining images into connected components of pHash distance ≤ `group_hamming` so unreviewed near-duplicates cannot straddle splits.
4. `StratifiedGroupKFold(7)` → 5 folds train, 1 val, 1 test (≈ 71/14/14 %).

In [38]:
# ============================================================
# 1a. Rice manifest
# ============================================================
import networkx as nx

def build_rice_manifest(group_hamming=2, seed=42):
    df = pd.read_csv(CSV_PATH)
    df["relative_path"] = df["label"] + "/" + df["image_id"]

    reasons = {}
    ex = pd.read_csv(AUDIT_DIR / "rice_exact_duplicate_exclusion_manifest.csv")
    for p in ex.loc[ex.decision == "exclude_exact_duplicate", "relative_path"]:
        reasons[p] = "exact_duplicate"

    cc = pd.read_csv(AUDIT_DIR / "rice_cross_class_near_duplicate_review.csv")
    cc = cc[cc.visual_review_decision == "confirmed_duplicate_by_user"]
    for p in pd.concat([cc.path_a, cc.path_b]):
        reasons.setdefault(p, "cross_class_label_conflict")

    sc = pd.read_csv(AUDIT_DIR / "near_duplicate_review" / "same_class_batch_01_reviewed.csv")
    sc = sc[sc.visual_review_decision == "same_physical_image"]
    for p in sc.path_b:
        reasons.setdefault(p, "same_class_near_duplicate")

    df["exclusion_reason"] = df["relative_path"].map(reasons)
    kept = df[df.exclusion_reason.isna()].copy().reset_index(drop=True)

    # leakage groups from perceptual-hash neighbours among kept images
    cand = pd.read_csv(AUDIT_DIR / "rice_near_duplicate_candidates.csv")
    cand = cand[cand.hamming_distance <= group_hamming]
    keep_set = set(kept.relative_path)
    G = nx.Graph()
    G.add_nodes_from(kept.relative_path)
    G.add_edges_from((a, b) for a, b in zip(cand.path_a, cand.path_b)
                     if a in keep_set and b in keep_set)
    gid = {}
    for i, comp in enumerate(nx.connected_components(G)):
        for n in comp: gid[n] = i
    kept["group"] = kept.relative_path.map(gid)

    # 7 folds: fold 0 → test, fold 1 → val, rest → train
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    kept["fold"] = -1
    for f, (_, idx) in enumerate(sgkf.split(kept, kept.label, kept.group)):
        kept.loc[idx, "fold"] = f
    kept["split"] = np.where(kept.fold == 0, "test", np.where(kept.fold == 1, "val", "train"))
    return df, kept

rice_all, rice_df = build_rice_manifest(cfg.group_hamming, cfg.seed)
RICE_CLASSES = sorted(rice_df.label.unique())
RICE_L2I = {c: i for i, c in enumerate(RICE_CLASSES)}

print("Original records :", len(rice_all))
print("Excluded         :", int(rice_all.exclusion_reason.notna().sum()))
print(rice_all.exclusion_reason.value_counts().to_string())
print("Kept             :", len(rice_df), "| leakage groups:", rice_df.group.nunique())
print("\nSplit × class:")
display(pd.crosstab(rice_df.label, rice_df.split)[["train", "val", "test"]])

# Leakage self-check: no group may appear in more than one split
chk = rice_df.groupby("group").split.nunique()
assert (chk == 1).all(), "group leakage across splits!"
print("\nGroup-leakage check passed (every group lives in exactly one split).")
rice_df.to_csv(EXP_DIR / "manifests" / "rice_manifest.csv", index=False)

Original records : 10407
Excluded         : 133
exclusion_reason
exact_duplicate               74
cross_class_label_conflict    39
same_class_near_duplicate     20
Kept             : 10274 | leakage groups: 9563

Split × class:


split,train,val,test
label,,,
bacterial_leaf_blight,327,65,64
bacterial_leaf_streak,266,54,51
bacterial_panicle_blight,233,53,50
blast,1225,242,251
brown_spot,680,132,139
dead_heart,1018,202,209
downy_mildew,432,89,87
hispa,1117,241,223
normal,1252,251,243



Group-leakage check passed (every group lives in exactly one split).


## 2. Data layer — Pearl-millet manifest

* Only `.jpg` files in the 5 class folders (PDFs / `.DS_Store` / `Audit` are ignored).
* Originals are up to 4000×3000; they are **EXIF-corrected and cached at ≤ 512 px** once, so every epoch is cheap and the cache is deterministic.
* pHash groups (distance ≤ 4, the threshold used in the audit) guard against any residual near-duplicates; `StratifiedGroupKFold(5)` produces the CV folds.

In [39]:
# ============================================================
# 2a. Pearl-millet manifest + 512px cache
# ============================================================
import imagehash

PEARL_FOLDERS = {
    "PESGL_DRECRO_exserohilum rostratum":   "exserohilum",
    "PESGL_MOESBU_smut":                    "smut",
    "PESGL_PYRISP_ Pyricularia sp":         "pyricularia",
    "PESGL_SCLPGR_Sclerospora graminicola": "sclerospora",
    "PESGL_healthy leaves":                 "healthy",
}
PEARL_CLASSES = sorted(PEARL_FOLDERS.values())
PEARL_L2I = {c: i for i, c in enumerate(PEARL_CLASSES)}

def build_pearl_manifest(n_folds=5, seed=42, max_side=512):
    rows = []
    for folder, short in PEARL_FOLDERS.items():
        for p in sorted((PEARL_DIR / folder).rglob("*.jpg")):
            rows.append({"label": short, "src": str(p), "name": p.name})
    df = pd.DataFrame(rows)

    cache_paths, hashes = [], []
    PEARL_CACHE.mkdir(parents=True, exist_ok=True)
    for r in df.itertuples():
        dst = PEARL_CACHE / r.label / r.name
        dst.parent.mkdir(exist_ok=True)
        if not dst.exists():
            with Image.open(r.src) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                im.thumbnail((max_side, max_side), Image.LANCZOS)
                im.save(dst, quality=95)
        cache_paths.append(str(dst))
        with Image.open(dst) as im:
            hashes.append(imagehash.phash(im))
    df["path"] = cache_paths

    # near-duplicate groups (pHash ≤ 4), same-class only
    G = nx.Graph(); G.add_nodes_from(range(len(df)))
    for i in range(len(df)):
        for j in range(i + 1, len(df)):
            if df.label[i] == df.label[j] and hashes[i] - hashes[j] <= 4:
                G.add_edge(i, j)
    gid = {}
    for k, comp in enumerate(nx.connected_components(G)):
        for n in comp: gid[n] = k
    df["group"] = df.index.map(gid)

    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df["fold"] = -1
    for f, (_, idx) in enumerate(sgkf.split(df, df.label, df.group)):
        df.loc[idx, "fold"] = f
    return df

pearl_df = build_pearl_manifest(cfg.n_folds, cfg.seed)
print("Pearl-millet images:", len(pearl_df), "| groups:", pearl_df.group.nunique())
display(pd.crosstab(pearl_df.label, pearl_df.fold, margins=True))
pearl_df.to_csv(EXP_DIR / "manifests" / "pearl_manifest.csv", index=False)

Pearl-millet images: 228 | groups: 223


fold,0,1,2,3,4,All
label,,,,,,
exserohilum,2,1,2,2,2,9
healthy,5,4,5,4,5,23
pyricularia,20,19,18,21,20,98
sclerospora,5,6,4,4,4,23
smut,15,16,16,14,14,75
All,47,46,45,45,45,228


## 2b. Specimen-aware folds (`fold_spec`) — leakage-reduced protocol

**Finding.** The pHash ≤ 4 grouping used for the original folds (`fold`, protocol *v1*) does not catch repeated photographs of the same leaf taken from slightly different framing (pHash distance 6–35, ImageNet-feature cosine 0.95–0.99; e.g. `PYRISP_64/65`, `DRECRO_3/4/5`, `SCLPGR_1/2`). Consecutive file numbers are frequently the same specimen, and these pairs were spread across v1 folds.

**Fix (new protocol, v1 left untouched).** Within each class, ImageNet ConvNeXt features (never rice-trained, so neither arm is favoured) are clustered by average-linkage at cosine ≥ 0.93; each cluster is a *specimen group*. `StratifiedGroupKFold` then builds `fold_spec`. Among 300 candidate shuffles, the one with no empty class-fold cell and the most balanced fold sizes is kept — selection uses **labels and groups only, never model results**. Everything downstream is run on both protocols: `v1` (original, `results.csv`) and `spec` (`results_spec.csv`).

In [ ]:
# ============================================================
# 2b. Specimen-aware folds (fold_spec)
# ============================================================
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

SPEC_TAU, SPEC_SHUFFLES = 0.93, 300
SPEC_CSV = EXP_DIR / "manifests" / "pearl_manifest_spec.csv"
SPEC_EMB = EXP_DIR / "manifests" / "pearl_imagenet_embeddings.npy"

@torch.no_grad()
def imagenet_embeddings(paths, size=288):
    """L2-normalised raw ConvNeXt-Tiny ImageNet features (no rice information)."""
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1).to(DEVICE).eval()
    tf = T.Compose([T.Resize((size, size)), T.ToTensor(),
                    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    out = []
    for p in paths:
        with Image.open(p) as im: x = tf(im.convert("RGB")).unsqueeze(0).to(DEVICE)
        out.append(F.normalize(m.avgpool(m.features(x)).flatten(1), dim=1).float().cpu())
    del m; torch.cuda.empty_cache()
    return torch.cat(out).numpy()

def build_spec_manifest(df, tau=SPEC_TAU, n_shuffles=SPEC_SHUFFLES, n_folds=5):
    E = np.load(SPEC_EMB) if SPEC_EMB.exists() else None
    if E is None or len(E) != len(df):
        E = imagenet_embeddings(df.path); np.save(SPEC_EMB, E)
    C = E @ E.T; n = len(df); g = np.zeros(n, int); k = 0
    for lab in df.label.unique():
        idx = np.where(df.label.values == lab)[0]
        if len(idx) == 1: g[idx] = k; k += 1; continue
        D = np.clip(1 - C[np.ix_(idx, idx)], 0, None); np.fill_diagonal(D, 0)
        cl = fcluster(linkage(squareform(D, checks=False), "average"), 1 - tau, "distance")
        g[idx] = cl + k; k += cl.max()
    best = None
    for rs in range(n_shuffles):                       # labels + groups only
        fold = -np.ones(n, int)
        sgkf = StratifiedGroupKFold(n_folds, shuffle=True, random_state=rs)
        for f, (_, te) in enumerate(sgkf.split(df, df.label, g)): fold[te] = f
        ct = pd.crosstab(df.label, fold); sz = np.bincount(fold, minlength=n_folds)
        score = (int((ct == 0).sum().sum()), int(sz.max() - sz.min()), int((ct.max(1) - ct.min(1)).max()))
        if best is None or score < best[0]: best = (score, rs, fold.copy())
    out = df[["path"]].copy()
    out["group_spec"], out["fold_spec"], out["spec_shuffle_seed"] = g, best[2], best[1]
    return out

if SPEC_CSV.exists():
    _spec = pd.read_csv(SPEC_CSV)                      # never regenerate once saved
else:
    _spec = build_spec_manifest(pearl_df); _spec.to_csv(SPEC_CSV, index=False)
_v1_fold = pearl_df["fold"].copy()
pearl_df = pearl_df.merge(_spec, on="path", how="left", validate="one_to_one")
assert pearl_df.fold_spec.notna().all() and (pearl_df["fold"] == _v1_fold).all()
pearl_df["fold_spec"] = pearl_df.fold_spec.astype(int)
assert (pearl_df.groupby("group_spec").fold_spec.nunique() == 1).all(), "specimen group straddles folds"

print("Specimen groups:", pearl_df.group_spec.nunique(), "| largest:", int(pearl_df.group_spec.value_counts().max()),
      "| shuffle seed:", int(pearl_df.spec_shuffle_seed.iloc[0]))
display(pd.crosstab(pearl_df.label, pearl_df.fold_spec, margins=True))

## 3. Datasets and augmentation

* Training: `RandomResizedCrop` (scale 0.4–1), flips (H + V — leaves have no canonical orientation), small rotation, colour jitter (kept mild: colour is diagnostic for lesions), ImageNet normalisation.
* Evaluation: deterministic resize to a square.
* Augmentation is identical for both domains so that differences come from initialisation, not preprocessing.

In [40]:
# ============================================================
# 3. Transforms, datasets, loaders
# ============================================================
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def make_transforms(img):
    train_tf = T.Compose([
        T.RandomResizedCrop((img, img), scale=(0.4, 1.0), ratio=(0.75, 1.33)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.RandomApply([T.RandomRotation(20)], p=0.5),
        T.ColorJitter(0.2, 0.2, 0.15, 0.02),
        T.ToTensor(), T.Normalize(MEAN, STD),
    ])
    eval_tf = T.Compose([T.Resize((img, img)), T.ToTensor(), T.Normalize(MEAN, STD)])
    return train_tf, eval_tf

class ImageDS(Dataset):
    """df must have columns: <path_col>, label."""
    def __init__(self, df, l2i, transform, path_col="path", root=None):
        self.paths = [str(Path(root) / p) if root else p for p in df[path_col]]
        self.y = np.array([l2i[l] for l in df.label], dtype=np.int64)
        self.tf = transform
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.tf(im.convert("RGB"))
        return x, int(self.y[i])

def make_loader(ds, train, cfg, sampler_weights=None):
    kw = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers,
              pin_memory=True, persistent_workers=cfg.num_workers > 0)
    if train and sampler_weights is not None:
        s = WeightedRandomSampler(sampler_weights, len(ds), replacement=True)
        return DataLoader(ds, sampler=s, drop_last=len(ds) >= cfg.batch_size, **kw)
    return DataLoader(ds, shuffle=train, drop_last=train and len(ds) >= cfg.batch_size, **kw)

def effective_number_weights(y, n_classes, beta=0.999):
    counts = np.bincount(y, minlength=n_classes).astype(float)
    w = (1 - beta) / (1 - beta ** np.maximum(counts, 1))
    w = w / w.sum() * n_classes
    return torch.tensor(w, dtype=torch.float32)

## 4. Model architecture — `CropDiseaseNet`

```
image (3×H×W)
  └─ features[0]  stem               ┐
     features[1]  stage 1 (96 ch)    │  depth-indexed blocks → layer-wise LR decay
     features[2]  downsample         │  and staged freezing
     features[3]  stage 2 (192 ch)   │
     features[4]  downsample          │
     features[5]  stage 3 (384 ch)   │
     features[6]  downsample         │
     features[7]  stage 4 (768 ch)   ┘
  └─ LayerNorm2d → global-avg-pool → Dropout → 768-d embedding
        ├── head["rice"]   Linear(768 → 10)
        └── head["pearl"]  Linear(768 → 5)      (added at transfer time)
```

**Freeze policies** for transfer

| policy | trainable | purpose |
|---|---|---|
| `linear_probe` | new head only | measures raw feature quality of the rice-pretrained backbone |
| `partial_ft` | blocks 6–7 + norm + head | adapts high-level features, keeps generic low-level filters |
| `full_ft` | everything, layer-wise LR decay | maximal adaptation |

Layer-wise LR decay: block *i* gets `lr · lld^(7−i)`; heads get `lr · head_lr_mult`.

In [41]:
# ============================================================
# 4. Model
# ============================================================
class CropDiseaseNet(nn.Module):
    N_BLOCKS = 8

    def __init__(self, backbone="convnext_tiny", imagenet=True, drop=0.3):
        super().__init__()
        if backbone != "convnext_tiny":
            raise ValueError("only convnext_tiny is wired for staged freezing")
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if imagenet else None
        try:
            m = models.convnext_tiny(weights=weights)
        except Exception as e:                       # offline / download failure
            print(f"[warn] ImageNet weights unavailable ({type(e).__name__}); using random init")
            m = models.convnext_tiny(weights=None)
        self.features = m.features
        self.norm = m.classifier[0]                  # LayerNorm2d
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(drop)
        self.embed_dim = 768
        self.heads = nn.ModuleDict()

    def add_head(self, name, n_classes):
        self.heads[name] = nn.Linear(self.embed_dim, n_classes)
        nn.init.trunc_normal_(self.heads[name].weight, std=0.02)
        nn.init.zeros_(self.heads[name].bias)
        return self

    def embed(self, x):
        x = self.features(x)
        x = self.pool(self.norm(x)).flatten(1)
        return x

    def forward(self, x, head):
        return self.heads[head](self.drop(self.embed(x)))

    # ---- freezing -----------------------------------------------------
    def set_trainable(self, policy, head):
        for p in self.parameters(): p.requires_grad = False
        def on(mod):
            for p in mod.parameters(): p.requires_grad = True
        on(self.heads[head])
        if policy == "linear_probe":
            pass
        elif policy == "partial_ft":
            on(self.features[6]); on(self.features[7]); on(self.norm)
        elif policy == "full_ft":
            on(self.features); on(self.norm)
        else:
            raise ValueError(policy)

    # ---- optimiser groups with layer-wise LR decay --------------------
    def param_groups(self, head, lr, wd, lld, head_mult):
        groups = []
        def add(mod, scale):
            ps = [p for p in mod.parameters() if p.requires_grad]
            if not ps: return
            dec = [p for p in ps if p.ndim > 1]; nodec = [p for p in ps if p.ndim <= 1]
            if dec:   groups.append({"params": dec,   "lr": lr * scale, "weight_decay": wd})
            if nodec: groups.append({"params": nodec, "lr": lr * scale, "weight_decay": 0.0})
        for i, blk in enumerate(self.features):
            add(blk, lld ** (self.N_BLOCKS - 1 - i))
        add(self.norm, 1.0)
        add(self.heads[head], head_mult)
        for g in groups: g["base_lr"] = g["lr"]
        return groups

def build_model(init, cfg, rice_ckpt=None):
    """init: 'scratch' | 'imagenet' | 'rice'  (rice = ImageNet → rice pre-trained, loaded from ckpt)
             'rice_scratch' = random → rice pre-trained (isolates the value of rice alone)"""
    m = CropDiseaseNet(cfg.backbone, imagenet=(init == "imagenet" or init == "rice"), drop=cfg.drop)
    if init in ("rice", "rice_scratch"):
        sd = torch.load(rice_ckpt, map_location="cpu")
        sd = {k: v for k, v in sd.items() if not k.startswith("heads.")}
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert not unexpected, unexpected
    return m

# quick architecture summary
_m = CropDiseaseNet(imagenet=False).add_head("rice", len(RICE_CLASSES)).add_head("pearl", len(PEARL_CLASSES))
print("Backbone params (M):", round(sum(p.numel() for p in _m.features.parameters()) / 1e6, 2))
print("Embedding dim:", _m.embed_dim, "| heads:", {k: v.out_features for k, v in _m.heads.items()})
del _m

Backbone params (M): 27.82
Embedding dim: 768 | heads: {'rice': 10, 'pearl': 5}


## 5. Training engine

* bf16 autocast, AdamW, per-step linear warm-up → cosine decay applied to every group's `base_lr`.
* Loss: cross-entropy with label smoothing and optional effective-number class weights; optional mixup.
* `fit()` supports validation-based early stopping on **macro-F1** (rice) or a fixed epoch budget with no selection (pearl-millet CV).
* `evaluate()` returns accuracy, balanced accuracy, macro-F1, per-class F1, confusion matrix, expected calibration error, plus raw logits for later analysis.

In [ ]:
# ============================================================
# 5. Engine
# ============================================================
def ece_score(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

@torch.no_grad()
def evaluate(model, head, loader, class_names, criterion=None):
    model.eval(); logits, ys, loss_sum, n = [], [], 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=cfg.amp and DEVICE.type == "cuda"):
            out = model(x, head)
        out = out.float()
        if criterion is not None: loss_sum += criterion(out, y).item() * len(y)
        n += len(y); logits.append(out.cpu()); ys.append(y.cpu())
    logits = torch.cat(logits); y = torch.cat(ys).numpy()
    probs = logits.softmax(1).numpy(); pred = probs.argmax(1)
    K = len(class_names); labels = list(range(K)); present = np.unique(y).tolist()
    pf1 = f1_score(y, pred, average=None, labels=labels, zero_division=0)
    prec, rec, _, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    absent = sup == 0
    return {
        "loss": loss_sum / max(n, 1),
        "acc": accuracy_score(y, pred),
        "bal_acc": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", labels=labels, zero_division=0),
        # macro-F1 over classes that actually occur in y (avoids a spurious 0 for an absent class)
        "macro_f1_present": f1_score(y, pred, average="macro", labels=present, zero_division=0),
        "per_class_f1": dict(zip(class_names, pf1)),
        "per_class_f1_present": dict(zip(class_names, np.where(absent, np.nan, pf1))),
        "per_class_prec": dict(zip(class_names, prec)),
        "per_class_rec": dict(zip(class_names, np.where(absent, np.nan, rec))),
        "confusion": confusion_matrix(y, pred, labels=labels),
        "ece": ece_score(probs, y),
        "y": y, "pred": pred, "probs": probs, "logits": logits.numpy(),
    }

def mixup(x, y, alpha, n_classes, smoothing):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def fit(model, head, train_ds, val_ds, class_names, *, epochs, lr, wd, patience=None,
        lld=1.0, head_mult=1.0, policy="full_ft", cfg=cfg, tag="run", verbose=True):
    n_classes = len(class_names)
    model.to(DEVICE).set_trainable(policy, head)
    groups = model.param_groups(head, lr, wd, lld, head_mult)
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))

    cw = effective_number_weights(train_ds.y, n_classes).to(DEVICE) if cfg.balance == "loss" else None
    sw = None
    if cfg.balance == "sampler":
        cnt = np.bincount(train_ds.y, minlength=n_classes); sw = (1.0 / np.maximum(cnt, 1))[train_ds.y]
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=cfg.label_smoothing)
    eval_crit = nn.CrossEntropyLoss()
    tl = make_loader(train_ds, True, cfg, sw)
    vl = make_loader(val_ds, False, cfg) if val_ds is not None else None

    total = max(1, epochs * len(tl)); warm = int(cfg.warmup_frac * total); step = 0
    hist, best, best_state, bad = [], -1.0, None, 0

    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run_loss, seen = 0.0, 0
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=cfg.amp and DEVICE.type == "cuda"):
                if cfg.mix_alpha > 0:
                    xm, ya, yb, lam = mixup(x, y, cfg.mix_alpha, n_classes, cfg.label_smoothing)
                    out = model(xm, head).float()
                    loss = lam * crit(out, ya) + (1 - lam) * crit(out, yb)
                else:
                    out = model(x, head).float(); loss = crit(out, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            opt.step(); step += 1
            run_loss += loss.item() * len(y); seen += len(y)
        row = {"epoch": ep, "train_loss": run_loss / seen, "sec": time.time() - t0}
        if vl is not None:
            r = evaluate(model, head, vl, class_names, eval_crit)
            row.update(val_loss=r["loss"], val_acc=r["acc"], val_macro_f1=r["macro_f1"])
            if r["macro_f1"] > best:
                best, bad = r["macro_f1"], 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
        hist.append(row)
        if verbose: print(f"[{tag}] ep {ep:02d}/{epochs} " + " ".join(f"{k}={v:.4f}" for k, v in row.items() if k != "epoch"))
        if patience and vl is not None and bad >= patience:
            print(f"[{tag}] early stop at epoch {ep}"); break

    if best_state is not None: model.load_state_dict(best_state)
    return pd.DataFrame(hist), best

## 6. Stage 1 — Supervised pre-training on rice

Trains `head["rice"]` + backbone, selecting the checkpoint with the best validation macro-F1. The held-out **test** split is evaluated once at the end. Two pre-training variants are supported so the transfer study can separate the contribution of ImageNet from that of rice:

* `rice` — ImageNet → rice (the main condition)
* `rice_scratch` — random → rice

In [ ]:
# ============================================================
# 6. Stage 1: rice pre-training
# ============================================================
def pretrain_rice(start="imagenet", cfg=cfg, max_train=None):
    name = "rice" if start == "imagenet" else "rice_scratch"
    ckpt = EXP_DIR / "checkpoints" / f"{name}_backbone.pt"
    if ckpt.exists() and os.environ.get("FORCE_OVERWRITE", "0") != "1":
        print(f"[skip] {ckpt.name} already exists - refusing to overwrite (FORCE_OVERWRITE=1 to retrain)")
        return None, ckpt, None
    seed_everything(cfg.seed)
    train_tf, eval_tf = make_transforms(cfg.img_size)
    sub = lambda d: d.sample(max_train, random_state=0) if (SMOKE and max_train and len(d) > max_train) else d
    tr, va, te = (sub(rice_df[rice_df.split == s]) for s in ("train", "val", "test"))
    mk = lambda d, tf: ImageDS(d, RICE_L2I, tf, "relative_path", root=IMAGE_DIR)
    tr_ds, va_ds, te_ds = mk(tr, train_tf), mk(va, eval_tf), mk(te, eval_tf)

    model = CropDiseaseNet(cfg.backbone, imagenet=(start == "imagenet"), drop=cfg.drop).add_head("rice", len(RICE_CLASSES))
    hist, best = fit(model, "rice", tr_ds, va_ds, RICE_CLASSES, epochs=cfg.rice_epochs, lr=cfg.rice_lr,
                     wd=cfg.rice_wd, patience=cfg.rice_patience, lld=cfg.lld, head_mult=1.0,
                     policy="full_ft", cfg=cfg, tag=f"rice[{start}]")
    torch.save(model.state_dict(), ckpt)

    res = evaluate(model, "rice", make_loader(te_ds, False, cfg), RICE_CLASSES)
    print(f"\nRICE TEST  acc={res['acc']:.4f}  bal_acc={res['bal_acc']:.4f}  macro_F1={res['macro_f1']:.4f}  ECE={res['ece']:.4f}")
    hist.to_csv(EXP_DIR / f"{name}_history.csv", index=False)
    pd.DataFrame({"class": RICE_CLASSES, "f1": [res["per_class_f1"][c] for c in RICE_CLASSES]}) \
        .to_csv(EXP_DIR / f"{name}_test_per_class_f1.csv", index=False)
    json.dump({"best_val_macro_f1": best, "test_acc": res["acc"], "test_bal_acc": res["bal_acc"],
               "test_macro_f1": res["macro_f1"], "test_ece": res["ece"], "cfg": asdict(cfg)},
              open(EXP_DIR / f"{name}_summary.json", "w"), indent=2)
    return model, ckpt, res

## 7. Stage 2 — Transfer to pearl millet (experiment grid)

For every `(init, policy, data_fraction, fold, seed)` a fresh model is built, trained for a **fixed** epoch budget on the training folds (optionally sub-sampled per class), and evaluated on the held-out fold.

| id | init | policy | question answered |
|---|---|---|---|
| `scratch_full` | random | full_ft | lower bound |
| `imagenet_linear` / `imagenet_partial` / `imagenet_full` | ImageNet | … | standard transfer baseline |
| `rice_linear` / `rice_partial` / `rice_full` | ImageNet→rice | … | **does rice add value over ImageNet?** |
| `ricescratch_full` | random→rice | full_ft | value of rice alone |

Data fractions (10/25/50/100 %) show whether rice pre-training matters most in the low-data regime. Results append to `results.csv`; finished runs are skipped on restart.

In [ ]:

# ============================================================
# 7. Stage 2: pearl-millet transfer grid  (protocols: v1 original folds, spec specimen-grouped folds)
# ============================================================
EXPERIMENTS = {
    "scratch_full":     dict(init="scratch",      policy="full_ft"),
    "imagenet_linear":  dict(init="imagenet",     policy="linear_probe"),
    "imagenet_partial": dict(init="imagenet",     policy="partial_ft"),
    "imagenet_full":    dict(init="imagenet",     policy="full_ft"),
    "rice_linear":      dict(init="rice",         policy="linear_probe"),
    "rice_partial":     dict(init="rice",         policy="partial_ft"),
    "rice_full":        dict(init="rice",         policy="full_ft"),
    "ricescratch_full": dict(init="rice_scratch", policy="full_ft"),
}
ALL_EXPS = list(EXPERIMENTS)
PROTOCOLS = {
    "v1":   dict(fold_col="fold",      results=EXP_DIR / "results.csv",      preds=EXP_DIR / "predictions.csv",      extended=False),
    "spec": dict(fold_col="fold_spec", results=EXP_DIR / "results_spec.csv", preds=EXP_DIR / "predictions_spec.csv", extended=True),
}
RESULTS_CSV = PROTOCOLS["v1"]["results"]

def ckpt_for(init):
    return {"rice": EXP_DIR / "checkpoints" / "rice_backbone.pt",
            "rice_scratch": EXP_DIR / "checkpoints" / "rice_scratch_backbone.pt"}.get(init)

def available_exps():
    return [e for e in ALL_EXPS if ckpt_for(EXPERIMENTS[e]["init"]) is None or ckpt_for(EXPERIMENTS[e]["init"]).exists()]

def subsample_per_class(df, frac, seed):
    """Per-class subsample (>= 2 images). Identical for every experiment with the same (fold, seed, frac)."""
    if frac >= 1.0: return df
    parts = [g.sample(min(len(g), max(2, int(round(len(g) * frac)))), random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts)

def _train_eval(exp_id, tr, te, cfg_, seed_val, tag, extra=None):
    """Train a fresh model on `tr`; evaluate on `te` (and optionally on `extra`, e.g. a calibration set)."""
    spec = EXPERIMENTS[exp_id]
    seed_everything(seed_val)
    train_tf, eval_tf = make_transforms(cfg_.img_size)
    tr_ds = ImageDS(tr, PEARL_L2I, train_tf); te_ds = ImageDS(te, PEARL_L2I, eval_tf)
    model = build_model(spec["init"], cfg_, ckpt_for(spec["init"])).add_head("pearl", len(PEARL_CLASSES))
    lr = cfg_.pearl_lr * (5 if spec["policy"] == "linear_probe" else 1)
    fit(model, "pearl", tr_ds, None, PEARL_CLASSES, epochs=cfg_.pearl_epochs, lr=lr, wd=cfg_.pearl_wd,
        lld=cfg_.lld if spec["policy"] == "full_ft" else 1.0, head_mult=cfg_.head_lr_mult,
        policy=spec["policy"], cfg=cfg_, tag=tag, verbose=False)
    r = evaluate(model, "pearl", make_loader(te_ds, False, cfg_), PEARL_CLASSES)
    rx = None
    if extra is not None:
        rx = evaluate(model, "pearl", make_loader(ImageDS(extra, PEARL_L2I, eval_tf), False, cfg_), PEARL_CLASSES)
    del model; torch.cuda.empty_cache()
    return r, rx

def make_row(exp_id, frac, fold, seed, tr, te, r, extended):
    spec = EXPERIMENTS[exp_id]
    row = dict(exp=exp_id, init=spec["init"], policy=spec["policy"], frac=frac, fold=fold, seed=seed,
               n_train=len(tr), n_test=len(te), acc=r["acc"], bal_acc=r["bal_acc"],
               macro_f1=r["macro_f1"], ece=r["ece"],
               **{f"f1_{c}": r["per_class_f1"][c] for c in PEARL_CLASSES})
    if extended:
        row["macro_f1_present"] = r["macro_f1_present"]
        for c in PEARL_CLASSES:
            row[f"f1p_{c}"] = r["per_class_f1_present"][c]
            row[f"prec_{c}"] = r["per_class_prec"][c]
            row[f"rec_{c}"] = r["per_class_rec"][c]
    return row

def _append(df, path):
    df.to_csv(path, mode="a", header=not Path(path).exists(), index=False)

def run_pearl(exp_id, fold, seed=0, frac=1.0, cfg=cfg, save_preds=True, protocol="v1", preds_path=None):
    P = PROTOCOLS[protocol]; col = P["fold_col"]
    tr = subsample_per_class(pearl_df[pearl_df[col] != fold], frac, seed)
    te = pearl_df[pearl_df[col] == fold]
    assert not (set(tr.path) & set(te.path)), "train/test overlap"
    r, _ = _train_eval(exp_id, tr, te, cfg, 1000 * seed + fold, f"{exp_id}|f{fold}|s{seed}|{frac}")
    row = make_row(exp_id, frac, fold, seed, tr, te, r, P["extended"])
    if save_preds:
        _append(pd.DataFrame({"path": te.path.values, "y": r["y"], "pred": r["pred"]}).assign(
            exp=exp_id, fold=fold, seed=seed, frac=frac), preds_path or P["preds"])
    return row

def run_grid(exp_ids, folds, seeds=(0,), fracs=(1.0,), protocol="v1", cfg=cfg, results_path=None, preds_path=None):
    """Resumable: finished (exp, fold, seed, frac) rows are skipped; results are only ever appended."""
    P = PROTOCOLS[protocol]; rp = Path(results_path or P["results"])
    def finished():
        return pd.read_csv(rp) if rp.exists() else pd.DataFrame(columns=["exp", "fold", "seed", "frac"])
    for fr in fracs:
        for s in seeds:
            for f in folds:
                for e in exp_ids:
                    d = finished()
                    if ((d.exp == e) & (d.fold == f) & (d.seed == s) & np.isclose(d.frac.astype(float), fr)).any():
                        continue
                    t0 = time.time(); row = run_pearl(e, f, s, fr, cfg=cfg, protocol=protocol, preds_path=preds_path)
                    _append(pd.DataFrame([row]), rp)
                    print(f"[{protocol}] {e:18s} fold={f} seed={s} frac={fr:.2f} "
                          f"macroF1={row.get('macro_f1_present', row['macro_f1']):.3f} acc={row['acc']:.3f} ({time.time() - t0:.0f}s)")


## 8. Stage 3 — Evaluation and analysis

* **Summary table**: mean ± std of macro-F1 / balanced accuracy / ECE per experiment (and per data fraction).
* **Paired comparison**: for the same (fold, seed), `rice_*` vs `imagenet_*` macro-F1 differences with a Wilcoxon signed-rank test and bootstrap CI. With ≤ 10 pairs, treat p-values as indicative only.
* **Embedding plot**: PCA of backbone features for rice and pearl millet — shows the domain gap.
* **Grad-CAM**: checks that the model attends to lesions rather than background.

In [45]:
# ============================================================
# 8a. Aggregation and paired comparison
# ============================================================
from scipy.stats import wilcoxon

def summarize(path=RESULTS_CSV):
    r = pd.read_csv(path)
    g = r.groupby(["exp", "frac"])[["macro_f1", "bal_acc", "acc", "ece"]].agg(["mean", "std", "count"])
    return r, g

def paired_rice_vs_imagenet(r, frac=1.0, metric="macro_f1", n_boot=5000, seed=0):
    rng = np.random.default_rng(seed); out = []
    for pol in ("linear", "partial", "full"):
        a = r[(r.exp == f"rice_{pol}") & np.isclose(r.frac, frac)].set_index(["fold", "seed"])[metric]
        b = r[(r.exp == f"imagenet_{pol}") & np.isclose(r.frac, frac)].set_index(["fold", "seed"])[metric]
        d = (a - b).dropna().values
        if len(d) < 3: continue
        boots = [rng.choice(d, len(d)).mean() for _ in range(n_boot)]
        p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        out.append(dict(policy=pol, n_pairs=len(d), mean_diff=d.mean(),
                        ci_lo=np.percentile(boots, 2.5), ci_hi=np.percentile(boots, 97.5), wilcoxon_p=p))
    return pd.DataFrame(out)

def plot_fraction_curves(r, metric="macro_f1"):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for e, g in r.groupby("exp"):
        s = g.groupby("frac")[metric].agg(["mean", "std"])
        if len(s) > 1: ax.errorbar(s.index * 100, s["mean"], s["std"], marker="o", capsize=3, label=e)
    ax.set_xlabel("% of pearl-millet training data"); ax.set_ylabel(metric); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(EXP_DIR / "figures" / f"fraction_curves_{metric}.png", dpi=150); return fig

def plot_confusion(cm, names, title, fname):
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.5, 5)); im = ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    ax.set_title(title); ax.set_xlabel("predicted"); ax.set_ylabel("true"); fig.tight_layout()
    fig.savefig(EXP_DIR / "figures" / fname, dpi=150); return fig

In [46]:
# ============================================================
# 8b. Embeddings (domain gap) and Grad-CAM
# ============================================================
@torch.no_grad()
def extract_embeddings(model, ds, cfg=cfg):
    model.to(DEVICE).eval(); E, Y = [], []
    for x, y in make_loader(ds, False, cfg):
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=cfg.amp and DEVICE.type == "cuda"):
            E.append(model.embed(x.to(DEVICE)).float().cpu())
        Y.append(y)
    return torch.cat(E).numpy(), torch.cat(Y).numpy()

def plot_domain_gap(model, n_rice=1500, cfg=cfg, fname="domain_gap_pca.png"):
    from sklearn.decomposition import PCA
    _, eval_tf = make_transforms(cfg.img_size)
    rs = rice_df.sample(min(n_rice, len(rice_df)), random_state=0)
    Er, _ = extract_embeddings(model, ImageDS(rs, RICE_L2I, eval_tf, "relative_path", root=IMAGE_DIR), cfg)
    Ep, yp = extract_embeddings(model, ImageDS(pearl_df, PEARL_L2I, eval_tf), cfg)
    Z = PCA(2, random_state=0).fit_transform(np.vstack([Er, Ep]))
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(Z[:len(Er), 0], Z[:len(Er), 1], s=6, c="lightgray", label="rice")
    for k, c in enumerate(PEARL_CLASSES):
        m = yp == k; ax.scatter(Z[len(Er):][m, 0], Z[len(Er):][m, 1], s=22, label=f"pearl:{c}")
    ax.legend(fontsize=7); ax.set_title("Backbone embedding PCA"); fig.tight_layout()
    fig.savefig(EXP_DIR / "figures" / fname, dpi=150); return fig

class GradCAM:
    def __init__(self, model, head):
        self.m, self.head, self.act, self.grad = model.eval(), head, None, None
        layer = model.features[-1]
        layer.register_forward_hook(lambda _, __, o: setattr(self, "act", o))
        layer.register_full_backward_hook(lambda _, gi, go: setattr(self, "grad", go[0]))
    def __call__(self, x, target=None):
        x = x.to(DEVICE).requires_grad_(True)
        out = self.m(x, self.head); t = out.argmax(1) if target is None else torch.tensor([target]).to(DEVICE)
        self.m.zero_grad(); out[0, t[0]].backward()
        w = self.grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), int(t[0])

def show_gradcam(model, df, l2i, names, n_per_class=2, cfg=cfg, fname="gradcam.png"):
    _, eval_tf = make_transforms(cfg.img_size); cam = GradCAM(model, "pearl")
    sel = df.groupby("label", group_keys=False).apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=0))
    fig, axes = plt.subplots(2, len(sel), figsize=(2.2 * len(sel), 4.6))
    for i, r in enumerate(sel.itertuples()):
        with Image.open(r.path) as im: im = im.convert("RGB").resize((cfg.img_size, cfg.img_size))
        x = eval_tf(im).unsqueeze(0); heat, pred = cam(x)
        axes[0, i].imshow(im); axes[0, i].set_title(f"{r.label}\npred:{names[pred]}", fontsize=7)
        axes[1, i].imshow(im); axes[1, i].imshow(heat, cmap="jet", alpha=0.45)
        for a in axes[:, i]: a.axis("off")
    fig.tight_layout(); fig.savefig(EXP_DIR / "figures" / fname, dpi=150); return fig

## 10. Integrity, leakage and reproducibility checks

`verify_integrity()` writes `Experiments/integrity/integrity_report.json` (+ CSVs) and checks:

1. **Rice** — kept images are byte-unique; no leakage group crosses train/val/test; how many pHash-near pairs (≤ 2 / 3–6) still cross splits.
2. **Pearl millet** — byte-unique; for **both** fold protocols, the share of test images that have a near-identical same-class *specimen* in the training folds (ImageNet-feature cosine). This is what exposes the v1 leakage.
3. **Cross-domain** — rice ↔ pearl-millet pHash overlap (pre-training data must not contain target images).
4. **Fair comparison** — every experiment with the same (protocol, fold, seed, fraction) saw the same training subset size and test fold; training and test sets are disjoint; out-of-fold predictions cover each image exactly once.
5. **Preservation** — files in `_snapshot_2026-10-14/` are still byte-for-byte a prefix of the live result files and the rice checkpoint hash is unchanged.
6. **Reproducibility** — `repro_check()` re-trains one stored run and reports the deviation from the stored metric (GPU non-determinism is expected to be small but non-zero).

In [ ]:
# ============================================================
# 10. Integrity / leakage / reproducibility
# ============================================================
import hashlib
INTEG_DIR = EXP_DIR / "integrity"; INTEG_DIR.mkdir(exist_ok=True)
SNAP_DIR = PROJECT_DIR / "Experiments" / "_snapshot_2026-10-14"

def _sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def _hex_to_u64(h): return np.array([int(x, 16) for x in h], dtype=np.uint64)
def _popcount64(x):
    return np.unpackbits(x.view(np.uint8).reshape(*x.shape, 8), axis=-1).sum(-1)

def specimen_similarity_report(fold_cols=("fold", "fold_spec")):
    """For each protocol: how many test images have a near-identical same-class image in the training folds?"""
    E = np.load(SPEC_EMB); C = E @ E.T; lab = pearl_df.label.values; rows = []
    for col in fold_cols:
        fold = pearl_df[col].values
        mx = np.zeros(len(pearl_df))
        for i in range(len(pearl_df)):
            m = (lab == lab[i]) & (fold != fold[i])
            mx[i] = C[i, m].max() if m.any() else np.nan
        pearl_df[f"_maxsim_{col}"] = mx
        for thr in (0.90, 0.93, 0.95, 0.97):
            rows.append(dict(protocol=col, cosine_threshold=thr, n_images=len(mx),
                             test_images_with_near_specimen_in_train=int((mx >= thr).sum()),
                             share=float((mx >= thr).mean())))
        rows.append(dict(protocol=col, cosine_threshold="median max-similarity", n_images=len(mx),
                         test_images_with_near_specimen_in_train=np.nan, share=float(np.nanmedian(mx))))
    return pd.DataFrame(rows)

def verify_integrity():
    rep = {}
    # ---- A. rice
    cand = pd.read_csv(AUDIT_DIR / "rice_near_duplicate_candidates.csv")
    split_of = dict(zip(rice_df.relative_path, rice_df.split))
    cand = cand[cand.path_a.isin(split_of) & cand.path_b.isin(split_of)].copy()
    cand["cross_split"] = cand.path_a.map(split_of) != cand.path_b.map(split_of)
    rep["rice_cross_split_phash_pairs"] = {f"d<={d}": int((cand.cross_split & (cand.hamming_distance <= d)).sum()) for d in (0, 2, 4, 6)}
    rep["rice_group_leakage"] = bool((rice_df.groupby("group").split.nunique() > 1).any())
    shas = [_sha256(IMAGE_DIR / p) for p in rice_df.relative_path]
    rep["rice_kept_images_byte_unique"] = len(set(shas)) == len(shas)
    rep["rice_kept_images"] = len(rice_df)
    # ---- B. pearl millet
    psha = [_sha256(p) for p in pearl_df.path]
    rep["pearl_images"] = len(pearl_df); rep["pearl_byte_unique"] = len(set(psha)) == len(psha)
    sim = specimen_similarity_report(); sim.to_csv(INTEG_DIR / "pearl_specimen_leakage_by_protocol.csv", index=False)
    rep["pearl_near_specimen_in_train"] = sim[sim.cosine_threshold.astype(str) == "0.95"][
        ["protocol", "test_images_with_near_specimen_in_train", "share"]].to_dict("records")
    # ---- C. rice <-> pearl contamination
    rh = pd.read_csv(AUDIT_DIR / "rice_perceptual_hashes.csv")
    ph = np.array([int(str(imagehash.phash(Image.open(p).convert("RGB"))), 16) for p in pearl_df.path], dtype=np.uint64)
    rv = _hex_to_u64(rh.phash.values)
    dist = _popcount64(rv[:, None] ^ ph[None, :])
    rep["rice_vs_pearl_min_phash_distance"] = int(dist.min())
    rep["rice_vs_pearl_pairs_phash_le6"] = int((dist <= 6).sum())
    # ---- D. fair comparison / coverage
    issues = []
    for proto, P in PROTOCOLS.items():
        if not P["results"].exists(): continue
        r = pd.read_csv(P["results"]).drop_duplicates(["exp", "fold", "seed", "frac"], keep="last")
        for (f, s, fr), g in r.groupby(["fold", "seed", "frac"]):
            if g.n_train.nunique() > 1 or g.n_test.nunique() > 1:
                issues.append(f"{proto}: unequal n_train/n_test for fold={f} seed={s} frac={fr}")
            tr = subsample_per_class(pearl_df[pearl_df[P["fold_col"]] != f], fr, s)
            te = pearl_df[pearl_df[P["fold_col"]] == f]
            if set(tr.path) & set(te.path): issues.append(f"{proto}: train/test overlap fold={f}")
            if len(tr) != g.n_train.iloc[0] or len(te) != g.n_test.iloc[0]:
                issues.append(f"{proto}: stored sizes differ from reconstruction fold={f} seed={s} frac={fr}")
        pr = pd.read_csv(P["preds"]).drop_duplicates(["exp", "fold", "seed", "frac", "path"], keep="last")
        for (e, s, fr), g in pr.groupby(["exp", "seed", "frac"]):
            if g.fold.nunique() == pearl_df[P["fold_col"]].nunique() and (len(g) != len(pearl_df) or g.path.nunique() != len(pearl_df)):
                issues.append(f"{proto}: OOF coverage wrong for {e} seed={s} frac={fr}")
        rep[f"{proto}_runs"] = int(len(r))
    rep["fairness_issues"] = issues
    # ---- E. preservation of the earlier results
    pres = {}
    for fn in ("results.csv", "predictions.csv"):
        a, b = (SNAP_DIR / fn), (PROJECT_DIR / "Experiments" / fn)
        if a.exists() and b.exists():
            pres[fn + "_snapshot_is_prefix_of_live"] = b.read_bytes().startswith(a.read_bytes())
    ck = PROJECT_DIR / "Experiments" / "checkpoints" / "rice_backbone.pt"
    snap = (SNAP_DIR / "SHA256SUMS.txt")
    if ck.exists() and snap.exists():
        want = [l.split()[0] for l in snap.read_text().splitlines() if l.endswith("rice_backbone.pt")]
        pres["rice_backbone.pt_hash_unchanged"] = bool(want) and _sha256(ck) == want[0]
    rep["preservation"] = pres
    json.dump(rep, open(INTEG_DIR / "integrity_report.json", "w"), indent=2, default=str)
    print(json.dumps(rep, indent=2, default=str)); display(sim)
    return rep

def repro_check(exp_id="imagenet_partial", fold=0, seed=0, frac=1.0):
    """Re-train one stored v1 run and compare with the stored metric."""
    stored = pd.read_csv(PROJECT_DIR / "Experiments" / "results.csv")
    stored = stored[(stored.exp == exp_id) & (stored.fold == fold) & (stored.seed == seed) & np.isclose(stored.frac, frac)].iloc[0]
    row = run_pearl(exp_id, fold, seed, frac, save_preds=False, protocol="v1")
    out = dict(exp=exp_id, fold=fold, seed=seed, stored_macro_f1=float(stored.macro_f1), rerun_macro_f1=float(row["macro_f1"]),
               abs_diff=abs(float(stored.macro_f1) - float(row["macro_f1"])), stored_acc=float(stored.acc), rerun_acc=float(row["acc"]))
    json.dump(out, open(INTEG_DIR / "repro_check.json", "w"), indent=2); print(out); return out

## 11. Statistics and publication figures

**Unit of analysis.** Folds share training data and seeds share folds, so the 15 runs per cell are *not* independent. Primary inference therefore collapses seeds → one value per fold (5 values) and reports

* mean paired difference and Cohen's *d<sub>z</sub>* (mean / SD of the per-fold differences),
* a *t* interval (df = 4) — wide, because n = 5,
* an exact sign-flip permutation test over folds (with 5 folds the smallest attainable two-sided p is 0.0625, so **it can never reach p < 0.05**; it is reported for completeness),
* folds won (out of 5).

A complementary **image-level paired bootstrap** on pooled out-of-fold predictions (stratified resampling of the 228 test images, same resample for both arms, averaged over seeds) captures test-set sampling uncertainty. It ignores fold-to-fold training variability, so it is optimistic on its own.

**Claim rule (conservative, pre-specified):** a difference is called *supported* only if the Holm-adjusted bootstrap p < 0.05, the bootstrap 95 % CI excludes 0, **and** the sign agrees in ≥ 4 of 5 folds. Otherwise it is reported as *inconclusive* — never as "no effect". Holm correction is applied within the primary family (rice vs ImageNet, 3 policies × fractions) of each protocol; all other comparisons are exploratory.

In [ ]:
# ============================================================
# 11a. Loading, summary tables, paired statistics
# ============================================================
from scipy import stats as sps
import itertools
TAB_DIR = EXP_DIR / "tables"; TAB_DIR.mkdir(exist_ok=True)
FIG_DIR = EXP_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
METRIC = "macro_f1_present"
ARM = lambda e: e.split("_")[0]
POL = lambda e: EXPERIMENTS[e]["policy"]
ARM_LABEL = {"scratch": "Random init", "imagenet": "ImageNet", "rice": "ImageNet → rice", "ricescratch": "Rice only (no ImageNet)"}
POL_LABEL = {"linear_probe": "Linear probe", "partial_ft": "Partial fine-tune", "full_ft": "Full fine-tune"}
FRACS_ALL = [0.1, 0.25, 0.5, 1.0]

def load_protocol(protocol):
    P = PROTOCOLS[protocol]
    r = pd.read_csv(P["results"]).drop_duplicates(["exp", "fold", "seed", "frac"], keep="last")
    if "macro_f1_present" not in r: r["macro_f1_present"] = r["macro_f1"]   # v1: every class is in every fold
    r["frac"] = r.frac.round(4); r["arm"] = r.exp.map(ARM); r["pol"] = r.exp.map(POL)
    pr = pd.read_csv(P["preds"]).drop_duplicates(["exp", "fold", "seed", "frac", "path"], keep="last")
    pr["frac"] = pr.frac.round(4)
    return r, pr

def fold_means(r, metric=METRIC):
    return r.groupby(["exp", "frac", "fold"])[metric].mean().unstack("fold")      # rows: (exp, frac), cols: fold

def t_ci(x, level=0.95):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; n = len(x)
    if n < 2: return (np.nan, np.nan)
    h = sps.t.ppf(0.5 + level / 2, n - 1) * x.std(ddof=1) / np.sqrt(n); return (x.mean() - h, x.mean() + h)

def summary_table(r, protocol):
    rows = []
    for (e, fr), g in r.groupby(["exp", "frac"]):
        fm = g.groupby("fold")[METRIC].mean(); lo, hi = t_ci(fm)
        rows.append(dict(protocol=protocol, exp=e, arm=ARM(e), policy=POL(e), frac=fr, n_runs=len(g), n_folds=g.fold.nunique(),
                         n_seeds=g.seed.nunique(), mean_n_train=g.n_train.mean(),
                         macro_f1=fm.mean(), macro_f1_sd_over_folds=fm.std(ddof=1), macro_f1_ci_lo=lo, macro_f1_ci_hi=hi,
                         macro_f1_sd_over_runs=g[METRIC].std(ddof=1),
                         bal_acc=g.groupby("fold").bal_acc.mean().mean(), acc=g.groupby("fold").acc.mean().mean(),
                         ece=g.groupby("fold").ece.mean().mean()))
    return pd.DataFrame(rows).sort_values(["frac", "policy", "arm"])

def oof_dict(pr):
    """(exp, frac, seed) -> (y, pred) aligned to sorted image paths; only complete out-of-fold sets."""
    order = sorted(pearl_df.path); out = {}
    for (e, fr, s), g in pr.groupby(["exp", "frac", "seed"]):
        if len(g) != len(order): continue
        g = g.set_index("path").loc[order]; out[(e, fr, s)] = (g.y.values, g.pred.values)
    return out, order

def _f1_codes(codes, K):
    cm = np.bincount(codes, minlength=K * K).reshape(K, K); tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn
    return float(np.mean(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)))

def per_class_and_confusion(oof, protocol):
    K = len(PEARL_CLASSES); pc, cmrows = [], []
    for (e, fr) in sorted({(k[0], k[1]) for k in oof}):
        seeds = sorted(k[2] for k in oof if k[0] == e and k[1] == fr); P_, R_, F_, CM = [], [], [], []
        for s in seeds:
            y, p = oof[(e, fr, s)]; cm = confusion_matrix(y, p, labels=range(K)); tp = np.diag(cm).astype(float)
            P_.append(tp / np.maximum(cm.sum(0), 1)); R_.append(tp / np.maximum(cm.sum(1), 1))
            F_.append(2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)); CM.append(cm)
        for k, c in enumerate(PEARL_CLASSES):
            pc.append(dict(protocol=protocol, exp=e, frac=fr, cls=c, n_seeds=len(seeds), support=int(CM[0][k].sum()),
                           precision=np.mean([x[k] for x in P_]), recall=np.mean([x[k] for x in R_]), f1=np.mean([x[k] for x in F_]),
                           f1_sd_over_seeds=np.std([x[k] for x in F_], ddof=1) if len(seeds) > 1 else np.nan))
        cmm = np.mean(CM, 0)
        for a, b in itertools.product(range(K), range(K)):
            cmrows.append(dict(protocol=protocol, exp=e, frac=fr, true=PEARL_CLASSES[a], pred=PEARL_CLASSES[b], mean_count_per_seed=cmm[a, b]))
    return pd.DataFrame(pc), pd.DataFrame(cmrows)

def holm(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p); adj = np.empty(n); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (n - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def signflip_p(d):
    d = np.asarray(d, float); n = len(d)
    signs = np.array(list(itertools.product([-1, 1], repeat=n)))
    return float((np.abs((signs * d).mean(1)) >= abs(d.mean()) - 1e-12).mean())

def paired_bootstrap(oof, a, b, frac, y_ref, n_boot=4000, seed=0):
    K = len(PEARL_CLASSES); rng = np.random.default_rng(seed)
    seeds = sorted({k[2] for k in oof if k[0] == a and k[1] == frac} & {k[2] for k in oof if k[0] == b and k[1] == frac})
    if not seeds: return None
    CA = [y_ref * K + oof[(a, frac, s)][1] for s in seeds]; CB = [y_ref * K + oof[(b, frac, s)][1] for s in seeds]
    cls_idx = [np.where(y_ref == k)[0] for k in range(K)]; diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([rng.choice(ix, len(ix)) for ix in cls_idx])
        diffs[i] = np.mean([_f1_codes(c[idx], K) for c in CA]) - np.mean([_f1_codes(c[idx], K) for c in CB])
    point = np.mean([_f1_codes(c, K) for c in CA]) - np.mean([_f1_codes(c, K) for c in CB])
    p = 2 * min((np.sum(diffs <= 0) + 1) / (n_boot + 1), (np.sum(diffs >= 0) + 1) / (n_boot + 1))
    return dict(pooled_diff=point, boot_lo=np.percentile(diffs, 2.5), boot_hi=np.percentile(diffs, 97.5), boot_p=min(1.0, p), n_seeds_pooled=len(seeds))

PRIMARY = [("rice_linear", "imagenet_linear"), ("rice_partial", "imagenet_partial"), ("rice_full", "imagenet_full")]
SECONDARY = [("ricescratch_full", "scratch_full"), ("ricescratch_full", "imagenet_full"), ("rice_full", "ricescratch_full")]

def paired_table(r, oof, protocol, n_boot=4000):
    fm = fold_means(r); rows = []
    y_ref = None
    for fam, comps in (("primary", PRIMARY), ("secondary", SECONDARY)):
        for a, b in comps:
            for fr in FRACS_ALL:
                if (a, fr) not in fm.index or (b, fr) not in fm.index: continue
                d = (fm.loc[(a, fr)] - fm.loc[(b, fr)]).dropna().values
                if len(d) < 2: continue
                lo, hi = t_ci(d)
                row = dict(protocol=protocol, family=fam, a=a, b=b, frac=fr, n_folds=len(d), fold_mean_diff=d.mean(),
                           fold_sd=d.std(ddof=1), cohens_dz=d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan,
                           t_ci_lo=lo, t_ci_hi=hi, folds_a_wins=int((d > 0).sum()), signflip_p=signflip_p(d))
                if y_ref is None:
                    ys = pd.read_csv(PROTOCOLS[protocol]["preds"], usecols=["path", "y"]).drop_duplicates("path").set_index("path").y
                    y_ref = ys.loc[sorted(pearl_df.path)].values
                bs = paired_bootstrap(oof, a, b, fr, y_ref, n_boot)
                if bs: row.update(bs)
                rows.append(row)
    t = pd.DataFrame(rows)
    if t.empty: return t
    t["signflip_p_holm"] = np.nan; t["boot_p_holm"] = np.nan
    for fam in t.family.unique():
        m = t.family == fam
        t.loc[m, "signflip_p_holm"] = holm(t.loc[m, "signflip_p"].values)
        if "boot_p" in t: t.loc[m, "boot_p_holm"] = holm(t.loc[m, "boot_p"].fillna(1).values)
    def claim(x):
        if "boot_p_holm" not in x or pd.isna(x.get("boot_p_holm")): return "inconclusive (no bootstrap)"
        sign_ok = (x.folds_a_wins >= 4 and x.pooled_diff > 0) or (x.folds_a_wins <= 1 and x.pooled_diff < 0)
        if x.boot_p_holm < 0.05 and (x.boot_lo > 0 or x.boot_hi < 0) and sign_ok:
            return f"supported: {x.a} {'>' if x.pooled_diff > 0 else '<'} {x.b}"
        return "inconclusive"
    t["claim"] = t.apply(claim, axis=1)
    return t

def build_tables(protocol, n_boot=4000):
    r, pr = load_protocol(protocol); oof, order = oof_dict(pr)
    s = summary_table(r, protocol); s.to_csv(TAB_DIR / f"summary_{protocol}.csv", index=False)
    pc, cm = per_class_and_confusion(oof, protocol)
    pc.to_csv(TAB_DIR / f"per_class_metrics_{protocol}.csv", index=False); cm.to_csv(TAB_DIR / f"confusion_{protocol}.csv", index=False)
    pt = paired_table(r, oof, protocol, n_boot); pt.to_csv(TAB_DIR / f"paired_stats_{protocol}.csv", index=False)
    # run-level per-class table straight from results (fold-level, noisy but complete)
    cols = [c for c in r.columns if c.startswith(("f1_", "f1p_", "prec_", "rec_"))]
    r.groupby(["exp", "frac"])[cols].mean().reset_index().to_csv(TAB_DIR / f"per_class_foldlevel_{protocol}.csv", index=False)
    return r, pr, oof, s, pc, cm, pt

In [ ]:
# ============================================================
# 11b. Publication-quality figures
# ============================================================
C_ARM = {"scratch": "#6250d6", "imagenet": "#2a78d6", "rice": "#eb6834", "ricescratch": "#1baf7a"}   # validated categorical palette (slots 7,1,2,3)
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
MARK = {"scratch": "s", "imagenet": "o", "rice": "^", "ricescratch": "D"}

def set_pub_style():
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 300, "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5,
        "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "font.family": "DejaVu Sans",
        "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.8, "axes.labelcolor": INK2, "text.color": INK,
        "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
        "figure.facecolor": "white", "axes.facecolor": "#fcfcfb", "pdf.fonttype": 42,
    })
set_pub_style()

def _save(fig, name):
    for ext in ("png", "pdf"): fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight")
    return fig

def fig_learning_curves(r, protocol, metric=METRIC):
    fm = fold_means(r, metric); fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.5), sharey=True)
    for ax, pol in zip(axes, ["linear_probe", "partial_ft", "full_ft"]):
        arms = ["imagenet", "rice"] + (["scratch", "ricescratch"] if pol == "full_ft" else [])
        for arm in arms:
            e = {"linear_probe": f"{arm}_linear", "partial_ft": f"{arm}_partial", "full_ft": f"{arm}_full"}[pol]
            xs, ms, los, his = [], [], [], []
            for fr in FRACS_ALL:
                if (e, fr) not in fm.index: continue
                v = fm.loc[(e, fr)].dropna().values; lo, hi = t_ci(v)
                xs.append(fr * 100); ms.append(v.mean()); los.append(lo); his.append(hi)
                ax.scatter([fr * 100] * len(v), v, s=7, color=C_ARM[arm], alpha=0.28, linewidths=0, zorder=2)
            if xs:
                ax.plot(xs, ms, color=C_ARM[arm], marker=MARK[arm], ms=5, lw=1.8, label=ARM_LABEL[arm], zorder=3,
                        markeredgecolor="#fcfcfb", markeredgewidth=0.8)
                ax.fill_between(xs, los, his, color=C_ARM[arm], alpha=0.12, linewidth=0, zorder=1)
        ax.set_xscale("log"); ax.set_xticks([10, 25, 50, 100]); ax.set_xticklabels(["10", "25", "50", "100"])
        ax.set_xlabel("Pearl-millet training data (%)"); ax.set_title(POL_LABEL[pol], color=INK, loc="left")
    axes[0].set_ylabel("Macro-F1 (mean over folds)")
    h, l = {}, {}
    for ax in axes:
        for hh, ll in zip(*ax.get_legend_handles_labels()): h[ll] = hh
    fig.legend(list(h.values()), list(h.keys()), loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.09))
    fig.suptitle(f"Learning curves — protocol: {'specimen-grouped folds' if protocol == 'spec' else 'original folds (v1)'}", x=0.01, ha="left", fontsize=10)
    fig.text(0.01, -0.15, "Lines: mean of 5 fold-level values (seeds averaged); band: 95 % t-interval over folds; dots: individual folds.", fontsize=7.5, color=INK2)
    fig.tight_layout(); return _save(fig, f"learning_curves_{protocol}")

def fig_paired_deltas(pt, protocol):
    t = pt[(pt.family == "primary")]; fig, ax = plt.subplots(figsize=(6.4, 3.6))
    cols = {"linear_probe": "#2a78d6", "partial_ft": "#eb6834", "full_ft": "#1baf7a"}
    pol_of = {"rice_linear": "linear_probe", "rice_partial": "partial_ft", "rice_full": "full_ft"}
    xs = np.arange(len(FRACS_ALL))
    for k, (a, pol) in enumerate(pol_of.items()):
        g = t[t.a == a].set_index("frac"); off = (k - 1) * 0.22
        for xi, fr in enumerate(FRACS_ALL):
            if fr not in g.index: continue
            x = g.loc[fr]
            ax.errorbar(xi + off, x.pooled_diff, yerr=[[x.pooled_diff - x.boot_lo], [x.boot_hi - x.pooled_diff]], fmt="o", ms=5.5,
                        color=cols[pol], capsize=2.5, lw=1.4, markeredgecolor="#fcfcfb", label=POL_LABEL[pol] if xi == 0 else None, zorder=3)
            ax.plot([xi + off] * 2, [x.t_ci_lo, x.t_ci_hi], color=cols[pol], lw=3, alpha=0.16, solid_capstyle="butt", zorder=1)
    ax.axhline(0, color=INK2, lw=0.9); ax.set_xticks(xs); ax.set_xticklabels([f"{int(f*100)} %" for f in FRACS_ALL])
    ax.set_xlabel("Pearl-millet training data"); ax.set_ylabel("Δ macro-F1  (ImageNet→rice − ImageNet)")
    ax.set_title(f"Effect of adding rice pre-training — {'specimen-grouped' if protocol == 'spec' else 'original'} folds", loc="left")
    ax.legend(frameon=False, loc="best")
    fig.text(0.01, -0.04, "Markers + whiskers: pooled out-of-fold difference with image-level bootstrap 95 % CI. Wide pale bars: fold-level t-interval (n = 5).", fontsize=7.5, color=INK2)
    fig.tight_layout(); return _save(fig, f"paired_deltas_{protocol}")

EXP_ORDER = ["scratch_full", "imagenet_linear", "imagenet_partial", "imagenet_full", "rice_linear", "rice_partial", "rice_full", "ricescratch_full"]
def fig_confusions(cm, protocol, frac):
    if cm.empty: return None
    g = cm[np.isclose(cm.frac, frac)]; K = len(PEARL_CLASSES)
    fig, axes = plt.subplots(2, 4, figsize=(12.5, 7.6), gridspec_kw=dict(wspace=0.08, hspace=0.62)); im = None
    for ax, e in zip(axes.ravel(), EXP_ORDER):
        d = g[g.exp == e]; ax.grid(False)
        if d.empty: ax.axis("off"); continue
        M = d.pivot(index="true", columns="pred", values="mean_count_per_seed").loc[PEARL_CLASSES, PEARL_CLASSES].values
        Mn = M / np.maximum(M.sum(1, keepdims=True), 1e-9)
        im = ax.imshow(Mn, vmin=0, vmax=1, cmap="Blues")
        for i in range(K):
            for j in range(K):
                if M[i, j] >= 0.05: ax.text(j, i, f"{M[i, j]:.0f}" if M[i, j] >= 10 else f"{M[i, j]:.1f}", ha="center", va="center", fontsize=6.5,
                                            color="white" if Mn[i, j] > 0.55 else INK)
        ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(PEARL_CLASSES, rotation=60, ha="right", fontsize=6.5)
        ax.set_yticklabels(PEARL_CLASSES if e in ("scratch_full", "rice_linear") else [], fontsize=6.5)
        ax.set_title(f"{ARM_LABEL[ARM(e)]}\n{POL_LABEL[POL(e)]}", fontsize=7.5, loc="left")
    fig.supxlabel("predicted", fontsize=8.5, y=0.0); fig.supylabel("true", fontsize=8.5, x=0.0)
    if im is not None: fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01, label="row-normalised")
    fig.suptitle(f"Pooled out-of-fold confusion — {int(frac*100)} % data, {'specimen-grouped' if protocol == 'spec' else 'original'} folds "
                 f"(cell text: mean count per seed)", x=0.01, ha="left", fontsize=9.5)
    return _save(fig, f"confusion_{protocol}_{int(round(frac*100)):03d}pct")

def fig_per_class_heatmap(pc, protocol):
    if pc.empty: return None
    fig, axes = plt.subplots(2, 2, figsize=(9, 7.2))
    for ax, fr in zip(axes.ravel(), FRACS_ALL):
        d = pc[np.isclose(pc.frac, fr)]; ax.grid(False)
        if d.empty: ax.axis("off"); continue
        M = d.pivot(index="exp", columns="cls", values="f1").reindex(index=[e for e in EXP_ORDER if e in set(d.exp)], columns=PEARL_CLASSES)
        im = ax.imshow(M.values, vmin=0, vmax=1, cmap="Blues")
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                v = M.values[i, j]
                if not np.isnan(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.6 else INK)
        ax.set_xticks(range(len(PEARL_CLASSES))); ax.set_xticklabels(PEARL_CLASSES, rotation=35, ha="right")
        ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(M.index, fontsize=7.5); ax.set_title(f"{int(fr*100)} % of target data", loc="left")
    fig.suptitle(f"Per-class F1 (pooled out-of-fold, mean over seeds) — {'specimen-grouped' if protocol == 'spec' else 'original'} folds", x=0.01, ha="left", fontsize=10)
    fig.tight_layout(); return _save(fig, f"per_class_f1_{protocol}")

def fig_protocol_comparison():
    """How much do the original (leaky) folds inflate each experiment at 100 % data?"""
    if not (PROTOCOLS["v1"]["results"].exists() and PROTOCOLS["spec"]["results"].exists()): return None
    s1 = pd.read_csv(TAB_DIR / "summary_v1.csv"); s2 = pd.read_csv(TAB_DIR / "summary_spec.csv")
    a = s1[np.isclose(s1.frac, 1.0)].set_index("exp"); b = s2[np.isclose(s2.frac, 1.0)].set_index("exp")
    exps = [e for e in EXP_ORDER if e in a.index and e in b.index]
    fig, ax = plt.subplots(figsize=(6.6, 3.8)); y = np.arange(len(exps))[::-1]
    for yi, e in zip(y, exps):
        ax.plot([b.loc[e, "macro_f1"], a.loc[e, "macro_f1"]], [yi, yi], color=GRID, lw=3, zorder=1)
    ax.scatter([a.loc[e, "macro_f1"] for e in exps], y, s=42, color="#2a78d6", label="original folds (v1)", zorder=3, edgecolor="#fcfcfb")
    ax.scatter([b.loc[e, "macro_f1"] for e in exps], y, s=42, color="#eb6834", marker="D", label="specimen-grouped folds", zorder=3, edgecolor="#fcfcfb")
    ax.set_yticks(y); ax.set_yticklabels(exps); ax.set_xlabel("Macro-F1 at 100 % data"); ax.legend(frameon=False, loc="lower left")
    ax.set_title("Effect of removing repeated-specimen leakage", loc="left"); fig.tight_layout()
    return _save(fig, "protocol_comparison_100pct")

def fig_rice_training(names=("rice", "rice_scratch")):
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.2)); k = 0
    for nm, col in zip(names, ["#2a78d6", "#1baf7a"]):
        p = EXP_DIR / f"{nm}_history.csv"
        if not p.exists(): continue
        h = pd.read_csv(p); lab = "ImageNet → rice" if nm == "rice" else "Random → rice"
        axes[0].plot(h.epoch, h.train_loss, color=col, lw=1.8, label=f"{lab} train"); axes[0].plot(h.epoch, h.val_loss, color=col, lw=1.8, ls="--", label=f"{lab} val")
        axes[1].plot(h.epoch, h.val_macro_f1, color=col, lw=1.8, marker="o", ms=3.5, label=lab); k += 1
    if not k: plt.close(fig); return None
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(frameon=False, fontsize=7); axes[0].set_title("Rice pre-training loss", loc="left")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation macro-F1"); axes[1].legend(frameon=False); axes[1].set_title("Rice validation macro-F1", loc="left")
    fig.tight_layout(); return _save(fig, "rice_training_curves")

def rice_test_report(name="rice"):
    ck = EXP_DIR / "checkpoints" / f"{name}_backbone.pt"
    if not ck.exists(): print("no checkpoint", ck); return None
    _, eval_tf = make_transforms(cfg.img_size)
    te = rice_df[rice_df.split == "test"]; ds = ImageDS(te, RICE_L2I, eval_tf, "relative_path", root=IMAGE_DIR)
    m = CropDiseaseNet(imagenet=False).add_head("rice", len(RICE_CLASSES)); m.load_state_dict(torch.load(ck, map_location="cpu")); m.to(DEVICE)
    res = evaluate(m, "rice", make_loader(ds, False, cfg), RICE_CLASSES); del m
    pd.DataFrame(res["confusion"], index=RICE_CLASSES, columns=RICE_CLASSES).to_csv(TAB_DIR / f"rice_test_confusion_{name}.csv")
    prec, rec, f1, sup = precision_recall_fscore_support(res["y"], res["pred"], labels=range(len(RICE_CLASSES)), zero_division=0)
    pd.DataFrame(dict(cls=RICE_CLASSES, precision=prec, recall=rec, f1=f1, support=sup)).to_csv(TAB_DIR / f"rice_test_per_class_{name}.csv", index=False)
    pd.DataFrame([dict(model=name, n_test=len(te), acc=res["acc"], bal_acc=res["bal_acc"], macro_f1=res["macro_f1"], ece=res["ece"])]) \
        .to_csv(TAB_DIR / f"rice_test_summary_{name}.csv", index=False)
    cm = res["confusion"]; Mn = cm / cm.sum(1, keepdims=True); K = len(RICE_CLASSES)
    fig, ax = plt.subplots(figsize=(6.2, 5.4)); ax.grid(False); im = ax.imshow(Mn, vmin=0, vmax=1, cmap="Blues")
    for i in range(K):
        for j in range(K):
            if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6.5, color="white" if Mn[i, j] > 0.55 else INK)
    ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(RICE_CLASSES, rotation=55, ha="right"); ax.set_yticklabels(RICE_CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Rice test confusion — {name} (macro-F1 {res['macro_f1']:.3f})", loc="left")
    fig.colorbar(im, ax=ax, fraction=0.04, label="row-normalised"); fig.tight_layout(); _save(fig, f"rice_test_confusion_{name}")
    return res

## 12. Controlled diagnostic studies (no test-fold contamination)

All studies use the **specimen-grouped folds** and **pre-specified variants that are all reported**; no variant is selected on test performance, and none feeds back into the main results.

### 12a. Exserohilum
Exserohilum has only 9 images (≈ 2 per fold). Hypotheses: (H1) too few / heterogeneous examples — three visually distinct sub-types; (H2) confusion with Pyricularia (leaf-spot look-alikes; ImageNet-feature cosine between `DRECRO_8/9` and several `PYRISP` images ≈ 0.97); (H3) loss weighting / resolution. `exserohilum_diagnostics()` needs no training (per-image correctness over all stored runs, confusion destinations, 1-NN separability in ImageNet space). `run_exserohilum_study()` then varies **one training factor at a time** (`sampler`: class-balanced sampling instead of loss weights; `nobal`: no balancing; `ls0`: no label smoothing; `res384`: higher resolution, 384 px vs 320 px (448 px does not fit in the ~14 GB of GPU memory left by another process at the same batch size)) against a same-code `base` run.

### 12b. Calibration
Expected calibration error was ≈ 0.2 (n ≈ 45 per fold, 15 bins, so it is noisy and biased upward). Calibration is studied with a **held-out calibration fold** `(test_fold + 1) mod 5` that is disjoint from the test fold: the model trains on the remaining 3 folds, a temperature is fitted on the calibration fold only, and the test fold is scored before and after. Variants separate label smoothing, class weighting and training length. Pooled out-of-fold ECE (all 228 images per seed), NLL, Brier score and confidence-vs-accuracy are reported.

### 12c. Steps-matched sensitivity
At 10–50 % data a fixed 40 epochs means fewer optimiser steps (≈ 40 vs 80). `run_steps_sensitivity()` repeats ImageNet vs rice with 80 epochs to check that conclusions do not depend on this.

In [ ]:
# ============================================================
# 12. Studies
# ============================================================
import dataclasses
STUDY_DIR = EXP_DIR / "studies"; STUDY_DIR.mkdir(exist_ok=True)
PROTO_STUDY = "spec"

def _done(path, keys):
    if not Path(path).exists(): return set()
    d = pd.read_csv(path); return set(map(tuple, d[keys].astype(str).values))

# ---------- 12a Exserohilum -------------------------------------------------
EXS_VARIANTS = {
    "base":    {},
    "sampler": dict(balance="sampler"),
    "nobal":   dict(balance="none"),
    "ls0":     dict(label_smoothing=0.0),
    "res384":  dict(img_size=384),
}
def run_exserohilum_study(inits=("imagenet", "rice"), folds=range(5), seeds=(0, 1, 2), variants=None):
    col = PROTOCOLS[PROTO_STUDY]["fold_col"]; runs = STUDY_DIR / "exserohilum_runs.csv"; preds = STUDY_DIR / "exserohilum_preds.csv"
    keys = ["variant", "init", "fold", "seed"]
    for v in (variants or EXS_VARIANTS):
        for init in inits:
            for s in seeds:
                for f in folds:
                    if (v, init, str(f), str(s)) in _done(runs, keys): continue
                    c = dataclasses.replace(cfg, **EXS_VARIANTS[v]); e = f"{init}_full"
                    tr = pearl_df[pearl_df[col] != f]; te = pearl_df[pearl_df[col] == f]; t0 = time.time()
                    r, _ = _train_eval(e, tr, te, c, 1000 * s + f, f"exs|{v}|{init}|f{f}|s{s}")
                    row = make_row(e, 1.0, f, s, tr, te, r, True); row.update(variant=v, init=init)
                    _append(pd.DataFrame({"path": te.path.values, "y": r["y"], "pred": r["pred"]}).assign(variant=v, init=init, fold=f, seed=s), preds)
                    _append(pd.DataFrame([row]), runs)
                    print(f"[exs] {v:8s} {init:8s} f{f} s{s} macroF1={row['macro_f1_present']:.3f} "
                          f"exs_rec={row['rec_exserohilum']} ({time.time() - t0:.0f}s)")

def exserohilum_diagnostics(protocols=("v1", "spec")):
    E = np.load(SPEC_EMB); C = E @ E.T; np.fill_diagonal(C, -1); lab = pearl_df.label.values
    ex_idx = np.where(lab == "exserohilum")[0]; rows = []
    for i in ex_idx:
        same = np.where(lab == lab[i])[0]; same = same[same != i]; other = np.where(lab != lab[i])[0]
        nn = np.argmax(C[i]); orow = dict(image=Path(pearl_df.path[i]).name, fold_v1=int(pearl_df.fold[i]), fold_spec=int(pearl_df.fold_spec[i]),
                                          width_height_cached=Image.open(pearl_df.path[i]).size,
                                          nearest_same_class_cos=float(C[i, same].max()), nearest_other_class_cos=float(C[i, other].max()),
                                          nearest_other_class_is=lab[other][np.argmax(C[i, other])], one_nn_label=lab[nn])
        for proto in protocols:
            P = PROTOCOLS[proto]
            if not P["preds"].exists(): continue
            pr = pd.read_csv(P["preds"]).drop_duplicates(["exp", "fold", "seed", "frac", "path"], keep="last")
            pr = pr[(pr.path == pearl_df.path[i]) & np.isclose(pr.frac, 1.0)]
            if pr.empty: continue
            yk = PEARL_L2I["exserohilum"]
            orow[f"{proto}_n_runs"] = len(pr); orow[f"{proto}_correct_rate"] = float((pr.pred == yk).mean())
            dest = pr.pred.map(dict(enumerate(PEARL_CLASSES))).value_counts(normalize=True)
            orow[f"{proto}_predicted_as"] = "; ".join(f"{k}:{v:.2f}" for k, v in dest.items())
            for arm in ("imagenet", "rice"):
                q = pr[pr.exp.str.startswith(arm)]; orow[f"{proto}_correct_rate_{arm}"] = float((q.pred == yk).mean()) if len(q) else np.nan
        rows.append(orow)
    d = pd.DataFrame(rows); d.to_csv(TAB_DIR / "exserohilum_image_diagnostics.csv", index=False)
    # 1-NN separability of every class in ImageNet feature space (leave-one-out)
    nn_lab = lab[np.argmax(C, axis=1)]
    sep = pd.DataFrame({"cls": PEARL_CLASSES, "n": [int((lab == c).sum()) for c in PEARL_CLASSES],
                        "one_nn_accuracy": [float((nn_lab[lab == c] == c).mean()) for c in PEARL_CLASSES]})
    sep.to_csv(TAB_DIR / "one_nn_separability_imagenet.csv", index=False)
    # figure: nine specimens with their stored outcomes
    fig, axes = plt.subplots(3, 3, figsize=(8.4, 8.0))
    for ax, (_, o) in zip(axes.ravel(), d.sort_values("image", key=lambda s: s.str.extract(r"(\d+)")[0].astype(int)).iterrows()):
        p = pearl_df.path[pearl_df.path.str.endswith(o.image)].iloc[0]; ax.imshow(Image.open(p)); ax.axis("off")
        col = "spec_correct_rate" if "spec_correct_rate" in o and not pd.isna(o.get("spec_correct_rate")) else "v1_correct_rate"
        ax.set_title(f"{o.image.replace('PESGL_', '').replace('.jpg', '')}  |  correct {o[col]:.0%}\nmostly → {str(o.get(col.replace('correct_rate', 'predicted_as'), ''))[:34]}", fontsize=7.5, loc="left")
    fig.suptitle("Exserohilum specimens: fraction of stored runs classified correctly", x=0.01, ha="left", fontsize=10); fig.tight_layout(); _save(fig, "exserohilum_specimens")
    display(d.drop(columns=["width_height_cached"], errors="ignore")); display(sep)
    return d, sep

def analyze_exserohilum_study():
    runs = pd.read_csv(STUDY_DIR / "exserohilum_runs.csv"); preds = pd.read_csv(STUDY_DIR / "exserohilum_preds.csv")
    K = len(PEARL_CLASSES); ex = PEARL_L2I["exserohilum"]; rows = []
    for (v, init), g in preds.groupby(["variant", "init"]):
        per_seed = []
        for s, gs in g.groupby("seed"):
            if gs.path.nunique() != len(pearl_df): continue
            cm = confusion_matrix(gs.y, gs.pred, labels=range(K)); tp = np.diag(cm).astype(float)
            per_seed.append(dict(exs_recall=tp[ex] / cm[ex].sum(), exs_precision=tp[ex] / max(cm[:, ex].sum(), 1),
                                 exs_f1=2 * tp[ex] / (cm[ex].sum() + cm[:, ex].sum()), macro_f1_pooled=_f1_codes((gs.y * K + gs.pred).values, K),
                                 exs_to_pyr=cm[ex, PEARL_L2I["pyricularia"]] / cm[ex].sum(), pyr_to_exs=cm[PEARL_L2I["pyricularia"], ex] / cm[PEARL_L2I["pyricularia"]].sum()))
        if per_seed:
            d = pd.DataFrame(per_seed); rows.append(dict(variant=v, init=init, n_seeds=len(d), **{k: d[k].mean() for k in d}, **{k + "_sd": d[k].std(ddof=1) for k in d}))
    out = pd.DataFrame(rows)
    if out.empty: print("exserohilum study: no complete out-of-fold sets yet"); return out, out
    out.to_csv(TAB_DIR / "exserohilum_study_summary.csv", index=False)
    # paired deltas vs base at fold level (same fold & seed)
    dl = []
    for init in runs.init.unique():
        b = runs[(runs.variant == "base") & (runs.init == init)].set_index(["fold", "seed"])
        for v in runs.variant.unique():
            if v == "base": continue
            x = runs[(runs.variant == v) & (runs.init == init)].set_index(["fold", "seed"])
            j = b.join(x, lsuffix="_b", rsuffix="_v", how="inner")
            for met in ("macro_f1_present", "rec_exserohilum", "prec_exserohilum", "f1p_exserohilum"):
                d = (j[f"{met}_v"] - j[f"{met}_b"]).groupby(level="fold").mean().dropna()
                if len(d) >= 2:
                    lo, hi = t_ci(d); dl.append(dict(init=init, variant=v, metric=met, n_folds=len(d), mean_delta=d.mean(), t_ci_lo=lo, t_ci_hi=hi, folds_better=int((d > 0).sum())))
    dl = pd.DataFrame(dl); dl.to_csv(TAB_DIR / "exserohilum_study_deltas_vs_base.csv", index=False)
    display(out.round(3)); display(dl.round(3)); return out, dl

# ---------- 12b Calibration -------------------------------------------------
CAL_VARIANTS = {
    "base":      dict(label_smoothing=0.1, balance="loss", pearl_epochs=40),
    "ls0":       dict(label_smoothing=0.0, balance="loss", pearl_epochs=40),
    "nobal":     dict(label_smoothing=0.1, balance="none", pearl_epochs=40),
    "ls0_nobal": dict(label_smoothing=0.0, balance="none", pearl_epochs=40),
    "ep80":      dict(label_smoothing=0.1, balance="loss", pearl_epochs=80),
}
def temperature_fit(logits, y):
    grid = np.exp(np.linspace(-1.5, 2.5, 321)); L = torch.tensor(logits, dtype=torch.float32); Y = torch.tensor(y)
    return float(grid[int(np.argmin([F.cross_entropy(L / t, Y).item() for t in grid]))])

def calib_metrics(logits, y, T=1.0):
    p = torch.tensor(logits / T).softmax(1).numpy(); conf = p.max(1); pred = p.argmax(1); K = p.shape[1]
    order = np.argsort(conf); ace = np.mean([abs((pred[ix] == y[ix]).mean() - conf[ix].mean()) for ix in np.array_split(order, 10)])
    return dict(acc=float((pred == y).mean()), mean_conf=float(conf.mean()), ece15=ece_score(p, y, 15), ece10=ece_score(p, y, 10), ace_equal_mass10=float(ace),
                nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None)).mean()), brier=float(((p - np.eye(K)[y]) ** 2).sum(1).mean()))

def run_calibration_study(inits=("imagenet", "rice"), folds=range(5), seeds=(0, 1, 2), variants=None):
    col = PROTOCOLS[PROTO_STUDY]["fold_col"]; runs = STUDY_DIR / "calibration_runs.csv"; lg = STUDY_DIR / "calibration_logits.csv"
    keys = ["variant", "init", "fold", "seed"]; nf = pearl_df[col].nunique()
    for v in (variants or CAL_VARIANTS):
        for init in inits:
            for s in seeds:
                for f in folds:
                    if (v, init, str(f), str(s)) in _done(runs, keys): continue
                    cf = (f + 1) % nf; c = dataclasses.replace(cfg, **CAL_VARIANTS[v]); e = f"{init}_full"
                    tr = pearl_df[(pearl_df[col] != f) & (pearl_df[col] != cf)]; ca = pearl_df[pearl_df[col] == cf]; te = pearl_df[pearl_df[col] == f]
                    assert not (set(tr.path) | set(ca.path)) & set(te.path); assert not set(tr.path) & set(ca.path); t0 = time.time()
                    r, rc = _train_eval(e, tr, te, c, 1000 * s + f, f"cal|{v}|{init}|f{f}|s{s}", extra=ca)
                    T = temperature_fit(rc["logits"], rc["y"]); b = calib_metrics(r["logits"], r["y"]); a = calib_metrics(r["logits"], r["y"], T)
                    row = dict(variant=v, init=init, fold=f, seed=s, calib_fold=cf, n_train=len(tr), n_calib=len(ca), n_test=len(te), T=T,
                               macro_f1_present=r["macro_f1_present"], **{f"before_{k}": x for k, x in b.items()}, **{f"after_{k}": x for k, x in a.items()})
                    L = pd.DataFrame(r["logits"], columns=[f"l{k}" for k in range(r["logits"].shape[1])]); L.insert(0, "path", te.path.values); L.insert(1, "y", r["y"])
                    _append(L.assign(variant=v, init=init, fold=f, seed=s, T=T), lg); _append(pd.DataFrame([row]), runs)
                    print(f"[cal] {v:10s} {init:8s} f{f} s{s} T={T:.2f} ECE {b['ece15']:.3f}→{a['ece15']:.3f} conf={b['mean_conf']:.2f} acc={b['acc']:.2f} ({time.time() - t0:.0f}s)")

def analyze_calibration(n_boot=2000):
    L = pd.read_csv(STUDY_DIR / "calibration_logits.csv"); lcols = [c for c in L.columns if c.startswith("l") and c[1:].isdigit()]; rows = []; rel = {}
    for (v, init, s), g in L.groupby(["variant", "init", "seed"]):
        if g.path.nunique() != len(pearl_df): continue
        lg, y = g[lcols].values, g.y.values
        # each image uses the temperature fitted on ITS calibration fold (stored per row as T)
        Tvec = g["T"].values[:, None]; p_raw = torch.tensor(lg).softmax(1).numpy(); p_ts = torch.tensor(lg / Tvec).softmax(1).numpy()
        for tag, p in (("raw", p_raw), ("temp_scaled", p_ts)):
            conf = p.max(1); order = np.argsort(conf)
            rows.append(dict(variant=v, init=init, seed=s, calibration=tag, acc=float((p.argmax(1) == y).mean()), mean_conf=float(conf.mean()),
                             ece15=ece_score(p, y, 15), ece10=ece_score(p, y, 10),
                             ace_equal_mass10=float(np.mean([abs((p.argmax(1)[ix] == y[ix]).mean() - conf[ix].mean()) for ix in np.array_split(order, 10)])),
                             nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None)).mean()), brier=float(((p - np.eye(p.shape[1])[y]) ** 2).sum(1).mean())))
            rel[(v, init, s, tag)] = (conf, (p.argmax(1) == y).astype(float))
    d = pd.DataFrame(rows)
    if d.empty: print("calibration: no complete out-of-fold logits yet"); return d, None
    d.to_csv(TAB_DIR / "calibration_pooled_by_seed.csv", index=False)
    summ = d.groupby(["variant", "init", "calibration"]).agg(["mean", "std"]).drop(columns=[]).round(4)
    summ.columns = ["_".join(c) for c in summ.columns]; summ = summ.reset_index(); summ.to_csv(TAB_DIR / "calibration_summary.csv", index=False)
    # figure 1: ECE by variant (raw vs temp-scaled)
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.5), sharey=True)
    for ax, init in zip(axes, ("imagenet", "rice")):
        sub = summ[summ.init == init]; vs = [v for v in CAL_VARIANTS if v in set(sub.variant)]; x = np.arange(len(vs)); w = 0.36
        for k, (tag, col) in enumerate((("raw", "#eb6834"), ("temp_scaled", "#2a78d6"))):
            q = sub[sub.calibration == tag].set_index("variant").loc[vs]
            ax.bar(x + (k - .5) * w, q.ece15_mean, w * 0.92, yerr=q.ece15_std, color=col, capsize=2, error_kw=dict(lw=1, ecolor=INK2), label="raw" if tag == "raw" else "temperature-scaled")
        ax.set_xticks(x); ax.set_xticklabels(vs, rotation=20); ax.set_title(f"{'ImageNet' if init == 'imagenet' else 'ImageNet → rice'}, full fine-tune", loc="left")
    axes[0].set_ylabel("Pooled out-of-fold ECE (15 bins)"); axes[0].legend(frameon=False); fig.tight_layout(); _save(fig, "calibration_ece_by_variant")
    # figure 2: reliability diagram for the base variant
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.8), sharey=True); edges = np.linspace(0, 1, 11)
    for ax, init in zip(axes, ("imagenet", "rice")):
        ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
        for tag, col in (("raw", "#eb6834"), ("temp_scaled", "#2a78d6")):
            cs, ac = [], []
            for (v, i2, s, t2), (c, a) in rel.items():
                if v == "base" and i2 == init and t2 == tag: cs.append(c); ac.append(a)
            if not cs: continue
            c = np.concatenate(cs); a = np.concatenate(ac); xs, ys, ns = [], [], []
            for lo, hi in zip(edges[:-1], edges[1:]):
                m = (c > lo) & (c <= hi)
                if m.sum() >= 5: xs.append(c[m].mean()); ys.append(a[m].mean()); ns.append(m.sum())
            ax.plot(xs, ys, marker="o", ms=4.5, color=col, lw=1.6, label="raw" if tag == "raw" else "temperature-scaled")
        ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.set_xlabel("confidence"); ax.set_title(f"{'ImageNet' if init == 'imagenet' else 'ImageNet → rice'} (base)", loc="left")
    axes[0].set_ylabel("accuracy"); axes[0].legend(frameon=False); fig.tight_layout(); _save(fig, "calibration_reliability_base")
    display(summ[["variant", "init", "calibration", "acc_mean", "mean_conf_mean", "ece15_mean", "ece15_std", "nll_mean", "brier_mean"]]); return d, summ

# ---------- 12c Steps-matched sensitivity -----------------------------------
def run_steps_sensitivity(exps=("imagenet_full", "rice_full", "imagenet_partial", "rice_partial"), folds=range(5), seeds=(0, 1, 2), fracs=(0.1, 0.25, 0.5)):
    c = dataclasses.replace(cfg, pearl_epochs=80)
    run_grid(list(exps), folds, seeds, fracs, protocol="spec", cfg=c, results_path=STUDY_DIR / "steps80_results_spec.csv",
             preds_path=STUDY_DIR / "steps80_predictions_spec.csv")

In [ ]:
# ============================================================
# 13. One-call analysis wrapper
# ============================================================
def run_full_analysis(protocols=("v1", "spec"), n_boot=4000, rice=True):
    out = {}
    for proto in protocols:
        if not PROTOCOLS[proto]["results"].exists(): print("no results for", proto); continue
        r, pr, oof, s, pc, cm, pt = build_tables(proto, n_boot)
        fig_learning_curves(r, proto);
        if not pt.empty: fig_paired_deltas(pt, proto)
        for fr in FRACS_ALL: fig_confusions(cm, proto, fr)
        fig_per_class_heatmap(pc, proto); plt.close("all")
        if cm.empty: print(f"[{proto}] no complete out-of-fold prediction sets yet - confusion/per-class/bootstrap skipped")
        out[proto] = dict(summary=s, per_class=pc, paired=pt)
        print(f"\n===== {proto}: summary ====="); display(s[["exp", "frac", "n_folds", "n_seeds", "macro_f1", "macro_f1_ci_lo", "macro_f1_ci_hi", "bal_acc", "acc", "ece"]].round(3))
        print(f"\n===== {proto}: paired (a − b) ====="); display(pt.round(4))
    fig_protocol_comparison(); fig_rice_training()
    if rice:
        for nm in ("rice", "rice_scratch"): rice_test_report(nm)
    plt.close("all"); return out

In [ ]:
# ============================================================
# 13b. Steps-matched sensitivity: do the conclusions change when 10-50 % runs get as many optimiser steps as 100 % runs?
# ============================================================
def analyze_steps_sensitivity():
    p80 = STUDY_DIR / "steps80_results_spec.csv"
    if not p80.exists(): print("no steps80 results yet"); return None
    a = pd.read_csv(p80).drop_duplicates(["exp", "fold", "seed", "frac"], keep="last")
    b = pd.read_csv(PROTOCOLS["spec"]["results"]).drop_duplicates(["exp", "fold", "seed", "frac"], keep="last")
    b = b[b.exp.isin(a.exp.unique()) & ~np.isclose(b.frac, 1.0)]
    rows = []
    for label, r in (("40 epochs (main)", b), ("80 epochs", a)):
        r = r.assign(frac=r.frac.round(4)); fm = r.groupby(["exp", "frac", "fold"])["macro_f1_present"].mean().unstack("fold")
        for pol in ("partial", "full"):
            for fr in (0.1, 0.25, 0.5):
                ke, kr = (f"imagenet_{pol}", fr), (f"rice_{pol}", fr)
                if ke not in fm.index or kr not in fm.index: continue
                d = (fm.loc[kr] - fm.loc[ke]).dropna(); lo, hi = t_ci(d)
                rows.append(dict(epochs=label, policy=pol, frac=fr, n_folds=len(d), imagenet=fm.loc[ke].mean(), rice=fm.loc[kr].mean(),
                                 diff_rice_minus_imagenet=d.mean(), t_ci_lo=lo, t_ci_hi=hi, folds_rice_wins=int((d > 0).sum())))
    out = pd.DataFrame(rows); out.to_csv(TAB_DIR / "steps_sensitivity_spec.csv", index=False); display(out.round(3)); return out

## 14. Orchestration

Flags are read from environment variables (default `0`), so every job can be launched headlessly **without editing the notebook**:

```bash
cd ~/cate_model/Pearl_millet_Prediction/Rice_Dataset/Helper
python nb_to_script.py Training_Notebook.ipynb /tmp/train_job.py
CUDA_VISIBLE_DEVICES=0 RUN_STAGE1_RICE_SCRATCH=1 RUN_SPEC_FULL=1 RUN_SPEC_FRACTIONS=1 python /tmp/train_job.py
```

| flag | job | output |
|---|---|---|
| `RUN_STAGE1_RICE[_SCRATCH]` | rice pre-training (never overwrites an existing checkpoint) | `checkpoints/`, `rice*_history.csv` |
| `RUN_V1_FULL`, `RUN_V1_FRACTIONS` | original-fold grid | `results.csv`, `predictions.csv` (append-only) |
| `RUN_SPEC_FULL`, `RUN_SPEC_FRACTIONS` | specimen-grouped grid | `results_spec.csv`, `predictions_spec.csv` |
| `RUN_EXSEROHILUM_STUDY`, `RUN_CALIBRATION_STUDY`, `RUN_STEPS_SENSITIVITY` | controlled diagnostics | `studies/` |
| `RUN_INTEGRITY`, `RUN_REPRO_CHECK` | leakage / fairness / preservation checks | `integrity/` |
| `RUN_ANALYSIS`, `RUN_STUDY_ANALYSIS` | tables, statistics, figures | `tables/`, `figures/` |

`SMOKE=1` runs a tiny end-to-end version into `Experiments_smoke/`.

In [ ]:
# ============================================================
# 14. Orchestrator  (flags are read from the environment first, so jobs can be launched headlessly;
#                    every flag defaults to False and every job is resumable / append-only)
# ============================================================
def flag(name, default=False):
    return os.environ.get(name, "1" if default else "0") == "1"

RUN_STAGE1_RICE         = flag("RUN_STAGE1_RICE")          # ImageNet → rice   (refuses to overwrite an existing checkpoint)
RUN_STAGE1_RICE_SCRATCH = flag("RUN_STAGE1_RICE_SCRATCH")  # random   → rice
RUN_V1_FULL             = flag("RUN_V1_FULL")              # original folds, 100 % data (adds missing ricescratch_full)
RUN_V1_FRACTIONS        = flag("RUN_V1_FRACTIONS")         # original folds, 10/25/50 %
RUN_SPEC_FULL           = flag("RUN_SPEC_FULL")            # specimen-grouped folds, 100 %
RUN_SPEC_FRACTIONS      = flag("RUN_SPEC_FRACTIONS")       # specimen-grouped folds, 10/25/50 %
RUN_EXSEROHILUM_STUDY   = flag("RUN_EXSEROHILUM_STUDY")
RUN_CALIBRATION_STUDY   = flag("RUN_CALIBRATION_STUDY")
RUN_STEPS_SENSITIVITY   = flag("RUN_STEPS_SENSITIVITY")
RUN_INTEGRITY           = flag("RUN_INTEGRITY")
RUN_REPRO_CHECK         = flag("RUN_REPRO_CHECK")
RUN_ANALYSIS            = flag("RUN_ANALYSIS")
RUN_STUDY_ANALYSIS      = flag("RUN_STUDY_ANALYSIS")
RUN_ORDER = os.environ.get("RUN_GPU_LIST", "")             # informational only

FOLDS  = [0] if (SMOKE and not flag("SMOKE_ALL_FOLDS")) else list(range(cfg.n_folds))
SEEDS  = (0,) if SMOKE else (0, 1, 2)
FRACS  = (0.5,) if SMOKE else (0.1, 0.25, 0.5)
if SMOKE:
    RUN_STAGE1_RICE = RUN_STAGE1_RICE_SCRATCH = RUN_V1_FULL = RUN_SPEC_FULL = RUN_SPEC_FRACTIONS = True
    RUN_EXSEROHILUM_STUDY = RUN_CALIBRATION_STUDY = RUN_INTEGRITY = RUN_ANALYSIS = RUN_STUDY_ANALYSIS = True

if RUN_STAGE1_RICE:         pretrain_rice("imagenet", max_train=256)
if RUN_STAGE1_RICE_SCRATCH:
    # random-init ConvNeXt needs far more epochs than the ImageNet-initialised run (15 epochs gave val macro-F1 0.36)
    _ep = int(os.environ.get("RICE_SCRATCH_EPOCHS", cfg.rice_epochs))
    pretrain_rice("scratch", cfg=dataclasses.replace(cfg, rice_epochs=_ep, rice_patience=max(cfg.rice_patience, 10) if _ep > cfg.rice_epochs else cfg.rice_patience), max_train=256)

exps = available_exps()
print("experiments available:", exps)
if RUN_V1_FULL:        run_grid(exps, FOLDS, SEEDS, (1.0,), protocol="v1")
if RUN_V1_FRACTIONS:   run_grid(exps, FOLDS, SEEDS, FRACS,  protocol="v1")
if RUN_SPEC_FULL:      run_grid(exps, FOLDS, SEEDS, (1.0,), protocol="spec")
if RUN_SPEC_FRACTIONS: run_grid(exps, FOLDS, SEEDS, FRACS,  protocol="spec")

if RUN_EXSEROHILUM_STUDY: run_exserohilum_study(folds=FOLDS, seeds=SEEDS, variants=(["base", "sampler"] if SMOKE else None))
if RUN_CALIBRATION_STUDY: run_calibration_study(folds=FOLDS, seeds=SEEDS, variants=(["base"] if SMOKE else None))
if RUN_STEPS_SENSITIVITY: run_steps_sensitivity(folds=FOLDS, seeds=SEEDS, fracs=FRACS, **({"exps": ("imagenet_full", "rice_full")} if SMOKE else {}))

if RUN_INTEGRITY:   verify_integrity()
if RUN_REPRO_CHECK: repro_check()
if RUN_ANALYSIS:    analysis = run_full_analysis(rice=True)
if RUN_STUDY_ANALYSIS:
    if (STUDY_DIR / "exserohilum_runs.csv").exists():
        exserohilum_diagnostics(); analyze_exserohilum_study()
    if (STUDY_DIR / "calibration_logits.csv").exists(): analyze_calibration()
    analyze_steps_sensitivity()
